# Lab 3 · The Scanner Mess

*Garbage in, garbage out: prove it, then fix it.*

**Your mission:** Find out what the new scanner export did to Garabato's digit reader, write the cleaners that fix it, and prove with numbers that clean data makes a better model.

**Reaching Putting It All Together (the end of the minimum path) is a complete lab. Everything after it is extra.**

**In this notebook**

1. Setup: run the first code cell
2. How this lab works: the markers ✏️ WRITE CODE and ▶️ JUST RUN, and the three parts CORE, OPTIONAL and ADVANCED
3. Previously on Garabato: a two-minute recap
4. How Garabato reads a postcode, and this month's news: three scanners
5. What is data quality? Garbage in, garbage out; encodings; a value or None; data minimisation
6. Just enough theory
7. Fast lane: the whole CORE on one table
8. Meet the data: the columns, last month vs this month, a first look at the whole file
9. The incident report, and the worked example W
10. CORE: C1 → C2 → C3 → Putting It All Together → C4
11. Check yourself: four exam-style questions on today's Python (5 minutes)
12. After the CORE: regular expressions and an AI red team (7–10 minutes)
13. OPTIONAL (extra practice): C5, ENG1, SCI1, BLD1
14. ADVANCED (for students who move faster): ENG2, SCI2, BLD2, SCI3 Does a real neural network care?, the Challenge: batch B (or compare your pipeline with pandas and DuckDB)
15. AI-use declaration, Before you go, What's next

Data: MNIST (LeCun, Cortes, Burges), CC BY-SA 3.0, adapted. Garabato Parcels is fictional.

**Today you review Theory S5–S6 (files, CSV and JSON, string methods, `try/except`, regular expressions, Unicode) by cleaning a messy scanner export; the ADVANCED task ENG2 also connects to Theory S13 (modules), by moving your cleaners into a module of your own.**

In [ ]:
# ▶️ JUST RUN · Run me first (Shift+Enter). You don't need to read this cell.
LAB = 3
KIT_URL = "https://github.com/xavitel/programming-lab/releases/download/lab3/lab3_kit.zip"   # filled in at build time if the kit is hosted; used only on Colab
import os, sys, platform
from pathlib import Path

def _find_kit(start):
    """Return the folder containing labkit/: here, up to 3 levels above, or 1 level below."""
    start = Path(start).resolve()
    for folder in [start] + list(start.parents)[:3]:
        if (folder / "labkit" / "__init__.py").is_file():
            return folder
    for child in sorted(p for p in start.iterdir() if p.is_dir()):
        if (child / "labkit" / "__init__.py").is_file():
            return child
    return None

print("Python", platform.python_version(), "|", sys.executable)
if sys.version_info < (3, 10):
    raise SystemExit("STOP: this kernel runs Python " + platform.python_version()
                     + ". The lab needs Python 3.10 or newer.\nFix: click the kernel name (top right) > "
                       "Select Another Kernel > Python Environments > pick 3.10+ (e.g. your course .venv). "
                       "START_HERE.md, Route A step 3, shows exactly where to click.")
if ".zip" in str(Path.cwd()).lower():
    raise SystemExit("STOP: the notebook was opened from inside the zip file.\nFix: close it, right-click "
                     "the zip > Extract All, then in VS Code use File > Open Folder on the extracted folder.")
IN_COLAB = "google.colab" in sys.modules
kit = _find_kit(Path.cwd())
if kit is None and IN_COLAB:
    import zipfile, urllib.request
    zips = sorted(Path("/content").glob("lab*_kit*.zip"))
    if not zips and KIT_URL:
        print("Colab: downloading the kit ...")
        urllib.request.urlretrieve(KIT_URL, "/content/kit.zip")
        zips = [Path("/content/kit.zip")]
    for z in zips:
        zipfile.ZipFile(z).extractall("/content")
    kit = _find_kit("/content")
if kit is None:
    raise SystemExit("STOP: I can't find the labkit folder.\nFix: open the WHOLE extracted kit folder in "
                     "VS Code (File > Open Folder), not only the notebook. On Colab, upload the kit zip "
                     "with the Files panel and run this cell again.")
os.chdir(kit)
if str(kit) not in sys.path:
    sys.path.insert(0, str(kit))
try:
    get_ipython().run_line_magic("load_ext", "autoreload")   # picks up edits in .py files
    get_ipython().run_line_magic("autoreload", "2")
except Exception:
    pass
import labkit
labkit.start(LAB)
from labkit import check, check_all, rescue, show_digit, side_by_side, load_digits

## How this lab works

**Two markers tell you what to do with every cell:**

| Marker | What it means |
|---|---|
| **✏️ WRITE CODE** | You write or change code here. Stubs start with `# ===== YOUR CODE HERE =====`: replace `pass` with your code. |
| **▶️ JUST RUN** | Provided code: run it (Shift+Enter) and read what it prints. You don't need to change it. |

Helper functions in `labkit` are already implemented: use them, don't modify them.

**Three parts:**

| Part | Who and when | Feedback |
|---|---|---|
| **CORE** | everyone, in order: C1 → C2 → C3 → Putting It All Together (the minimum path: reaching it is a complete lab), then C4 | `check()`, 4 hints, `rescue()` |
| **OPTIONAL** | extra practice when your CORE is done, any task, any order: C5, ENG1, SCI1, BLD1 | `check()` and hints |
| **ADVANCED** | for students who move faster; open-ended, you can continue at home: ENG2, SCI2, BLD2, SCI3 (a real neural network), the Challenge | a sanity check, a score, or your own measurements |

**Three tools.**
- `check("C1", clean_label)` answers PASS, NOT YET or NOT STARTED. It shows the tests it ran (TEST 1, TEST 2, …),
  with the inputs, the expected result and your result; on NOT YET, the first failing test and a likely cause.
- **Hints** sit under each task. Open them one at a time.
- `clean_label = rescue("C1")` hands you a working version, so the rest of the notebook keeps working.

**Stuck for 7 minutes after Hint 3? Use `rescue()`**, carry on, and come back to the task later.

**The AI dial** (a recommendation, not a ban):
- CORE: AI as **tutor**. Ask *why*, not for the answer; Hint 4 gives you a prompt. Try on your own for 7–10 minutes first.
- OPTIONAL and ADVANCED tasks: AI as **reviewer**. Keep only what you can explain line by line.
- The Challenge (ADVANCED): AI as **pair programmer**. Verify with checks and tests, and declare it at the end.
- Always: you own every line you run, you verify it, and you **never paste personal data into an AI tool**.

Work alone or with a neighbour, your choice. Nothing here is graded or collected.

## Previously on Garabato

Two minutes of recap. You don't need any code from earlier labs.

**The story so far:** Lab 1 *I can code* → Lab 2 *I can write good Python* → **Lab 3 (today)** *I can work with real data*

**Last time, in Lab 2:** you made the reader Pythonic without changing a single answer (`zip`, `sorted` with a key, counting with a dict), measured both versions and wrote your first `assert`s.

**The ideas you keep using:**

- An image is a list of 784 numbers (0 = paper, 255 = ink).
- Distance = the sum of squared differences.
- Nearest neighbour = copy the label of the closest known image.
- Average digit = one average image per digit (Lab 1 SCI1): the model today's incident retrains.
- Accuracy = the fraction right on images the model never saw.
- Missed a lab? Everything you need is in `labkit.reference`.

## How Garabato reads a postcode

Every parcel arrives with a handwritten label: the postcode, one digit per box, five boxes.

```text
the label on the parcel     the scanner cuts out the 5 boxes      the Postcode Machine reads them one by one
 [2][8][0][1][3]       ->    5 images of 28 x 28 pixels      ->    2, 8, 0, 1, 3  ->  route the parcel
```

- **One digit at a time.** The Postcode Machine (Labs 1 and 2) never reads a whole postcode at once: it reads
  one box, then the next. That is why every image in this lab is a single digit.
- **People still do the work.** Today the machine is not good enough on its own, so at the depot counter a
  clerk reads every label and **types the postcode** to route the parcel. That is slow, and it costs money.
- **The training data.** Every week a **labelling team** looks at a sample of box images, one image at a time,
  and **types the digit they see**. An image together with the digit a person typed is a *labelled example*.
- **Why train, if people already typed the answers?** Those typed digits belong to parcels that have already
  left. The machine learns from them to read the parcels nobody has typed yet. Every month Garabato retrains
  it on the new labelled examples, so that next month the clerks type fewer postcodes.

This month, the retraining went wrong.

## This month's news: three scanners

In November two depots got new scanners, from two new vendors. Here is what their spec sheets say:

| Scanner | Depot | Vendor | How it writes each pixel |
|---|---|---|---|
| **A** | Chamartín | the old one, unchanged | a whole number from 0 (paper) to 255 (ink): the format the model expects |
| **B** | Retiro | new | a whole number from 0 to 255, but **the other way round**: 255 is paper, 0 is ink (a negative) |
| **C** | Vallecas | new | a **fraction** from 0.0 (paper) to 1.0 (ink): `0.4627` instead of `118` |

Nobody told the data team, and the retraining script read the new export exactly as it read the old ones.
Keep this table in mind: in *Meet the data* you find out what each scanner did to the export.

## What is data quality?

The first export since the new scanners arrived, 1,500 box images to retrain the digit reader with, looks
fine at a glance. It isn't.

**Garbage in, garbage out.** A model learns whatever its training data says. If some images are stored the
wrong way round, some labels are typed as words and some rows are copies, the model learns those mistakes
too, and nothing crashes to warn you. **Data quality** means checking every value before you trust it, and
being able to say why you rejected the rest.

**Three ideas you'll use all lab:**

1. **Text has an encoding.** A file is bytes; `encoding="utf-8"` tells Python how to turn them into letters.
   Read UTF-8 bytes with the wrong encoding and `Chamartín` comes out as `ChamartÃ­n` (*mojibake*).
2. **A value or None.** A cleaner returns a clean value, or `None` when it can't trust its input. Why `None`?

   | What a label cleaner could do with `"7?"` | What happens next |
   |---|---|
   | crash (`int("7?")` raises `ValueError`) | the whole pipeline stops at that row: nothing gets cleaned |
   | guess (return `0`, or `-1`) | a wrong label slips into the training data, silently |
   | return `None` | the pipeline sees "I can't trust this", rejects the row and records why |

3. **Data minimisation** (GDPR). Keep only the data the purpose needs. To train a digit reader you need
   the image and its label. The e-mail address of the operator who typed the label, and free-text notes
   that sometimes hold a customer's phone number, are personal data the model never needs, so the clean
   data leaves them behind. Everything in this lab is fictional: the e-mails are pseudonyms at `example.org`, and the
   phone numbers come from ranges reserved for fiction. Still, treat them as if they were real: never paste
   them into an AI tool.

## Just enough theory

Open a primer only if the idea feels new. Each one is a short example you can run in your head.

<details>
<summary><strong>Reading a CSV file safely (Theory S5–S6)</strong></summary>

`open(path, mode, encoding=...)` opens a file. The **mode** says what you'll do with it: `"r"` reads (the
default); `"w"` writes: it creates the file, and if the file already exists it **empties it first**, so the old
content is gone. `with` closes the file for you, even after an error. `encoding` says how the bytes become
letters. `newline=""` hands the line ends to the csv module: it reads `\r\n` (Windows) and `\n` alike, keeps a
line break that sits inside a quoted value, and, when writing, avoids an empty line after every row on Windows.
`csv.DictReader` gives one dict per row, with the header's names as keys.

```python
import csv
with open("data/parcel_scans_raw.csv", encoding="utf-8", newline="") as f:
    for row in csv.DictReader(f):
        print(row["scan_id"], row["label_typed"])     # every value is a str
```

</details>

<details>
<summary><strong>String methods (Theory S6)</strong></summary>

They return a NEW string; the original never changes. `strip()` removes spaces (and tabs, line ends) at the
**two ends** only, never inside: `" 28 013 ".strip()` is `'28 013'`. To remove every space, `replace(" ", "")`.

```python
" SEVEN ".strip().lower()        # 'seven'
"0 12 255".split()               # ['0', '12', '255']   (no argument: any whitespace)
"28.013".replace(".", "")        # '28013'
"px-001".upper(), "E-28013".startswith("E-")    # 'PX-001', True
```

</details>

<details>
<summary><strong>A value or None, with try/except (Theory S6)</strong></summary>

`try` runs code that may fail; `except` says what to do if it does. Catch the error you expect, not every error.

```python
def to_int(text: str) -> int | None:
    try:
        return int(text)          # int(" 7") is 7: int() ignores spaces around the number
    except ValueError:            # "seven", "7.0", "NaN"
        return None
```

</details>

<details>
<summary><strong>Sets: have I seen this before? (Theory S3)</strong></summary>

A set holds each value once, and `in` answers instantly.

```python
seen = set()
for scan_id in ["PX-1", "PX-2", "PX-1"]:
    if scan_id in seen:
        print("duplicate:", scan_id)
    seen.add(scan_id)
```

</details>

<details>
<summary><strong>Counting with a dict (Theory S3)</strong></summary>

One key per thing you count. `.get(key, 0)` gives 0 for a key you haven't seen yet.

```python
reasons = {}
for reason in ["duplicate", "bad label", "duplicate"]:
    reasons[reason] = reasons.get(reason, 0) + 1
print(reasons)                  # {'duplicate': 2, 'bad label': 1}
```

</details>

<details>
<summary><strong>Writing CSV and JSON (Theory S5)</strong></summary>

`pathlib.Path` makes folders and paths. `mkdir(parents=True, exist_ok=True)` creates the folder (and any missing
folder above it); `exist_ok=True` means "fine if it is already there". With `exist_ok=False`, the default, the
second run of the same cell stops with `FileExistsError`. `csv.DictWriter` writes dicts as rows.
`json.dump(obj, f)` writes a Python object (dicts, lists, text, numbers, `True`, `None`) as JSON text into an
open file: `indent=2` puts each key on its own indented line, so a person can read it, and
`ensure_ascii=False` keeps `í` as `í` instead of `\u00ed`. Both write into a file you opened, never into a path.

```python
import csv, json
from pathlib import Path
folder = Path("output")
folder.mkdir(parents=True, exist_ok=True)      # create the folder; no error if it is already there
with open(folder / "out.csv", "w", encoding="utf-8", newline="") as f:   # folder / "out.csv": a path inside it
    writer = csv.DictWriter(f, fieldnames=["scan_id", "label"])
    writer.writeheader()
    writer.writerow({"scan_id": "PX-1", "label": 7})
with open(folder / "out.json", "w", encoding="utf-8") as f:
    json.dump({"valid": 1}, f, indent=2, ensure_ascii=False)
```

</details>

<details>
<summary><strong>Unicode and encodings in one minute (Theory S6)</strong></summary>

**Unicode** gives every character a number, its *code point*: `í` is U+00ED. An **encoding** stores those
numbers as bytes. **UTF-8** uses one byte for `a` and two for `í` (`c3 ad`). Older encodings such as
**ISO-8859-1** (Latin-1) and **Windows-1252** (cp1252, Windows' old default) use one byte per character and
know only about 250 characters. Read UTF-8 bytes as Latin-1 or cp1252 and each byte of `í` becomes a character
of its own: `Chamartín` turns into `ChamartÃ­n`. That is *mojibake*, and it hits any text with a character
outside plain English: names, addresses, `€`. The cell after this list shows it happening.

A letter such as `í` can also be written as TWO code points (an `i` plus a combining accent): it looks the same
but is not equal. `unicodedata.normalize("NFC", text)` turns both into one.

```python
import unicodedata
"Chamarti\u0301n" == "Chamartín"                                   # False
unicodedata.normalize("NFC", "Chamarti\u0301n") == "Chamartín"     # True
"Chamartín".encode("utf-8")          # b'Chamart\xc3\xadn': two bytes for í
```

</details>

In [ ]:
# ▶️ JUST RUN · Unicode, UTF-8 and mojibake, in five lines
word = "Chamartín"
print(len(word), "characters; the code point of í:", hex(ord("í")))
data = word.encode("utf-8")
print(len(data), "bytes in UTF-8:", data)
print("those bytes read as Latin-1:", repr(data.decode("latin-1")))
print("the same bytes read as UTF-8:", repr(data.decode("utf-8")))

## Fast lane

Some people like to read a spec and build; others like guided steps. Both routes reach the same place.

| Task | Signature | Spec |
|---|---|---|
| C1 (minimum path) | `clean_label(raw: str) -> int \| None` | strip and lower; one digit -> int; LABEL_WORDS or LOOK_ALIKES -> its digit; anything else -> None |
| C2 (minimum path) | `parse_pixels(raw: str) -> list[int] \| None` | split; exactly 784 pieces; each one int() (ValueError -> None) and from 0 to 255; else None |
| C3 (minimum path) | `fix_inversion(pixels: list[int]) -> list[int]` | mean above 127: [255 - p for p in pixels]; otherwise a copy; always a new list |
| C4 | `clean_scans(rows: list[dict[str, str]]) -> tuple[list[dict], dict]` | rules in order (duplicate, label, pixels; the first rule a row breaks gives its reason); keep only scan_id, depot, label, pixels; report {total, valid, invalid, reasons} |

(minimum path) marks the tasks you need for Putting It All Together: C1 → C2 → C3. Each function has a stub cell further down, starting with `# ===== YOUR CODE HERE =====`: write your code there, run it, then run the cell below at any time to see where you are. Each line tells you the next step.

In [ ]:
# ▶️ JUST RUN · where am I?
check_all();   # the ; only stops Jupyter from printing the same result again as a dict

## Meet the data

We'll work with **one scanner export of 1,500 rows**, `data/parcel_scans_raw.csv`: this month's box images for
the retraining. **One row is one box image** (one digit), not one parcel: the model learns one digit at a time,
so its training data comes one digit at a time. Every model we retrain is tested on the **same 2,000 test
images** (handwritten digits it never trains on).

Every column comes from somewhere, and where it comes from tells you what can go wrong with it:

| Column | Who or what writes it | Clean form | What can go wrong |
|---|---|---|---|
| `scan_id` | the scanner software: one id per box image | `PX-001234` | the same image exported twice, sometimes written `px-001234` or with spaces |
| `depot` | the depot's own system | `Chamartín`, `Retiro`, `Vallecas` | capitals, spaces, codes such as `CHM`, accents written two ways |
| `scanner` | the scanner software | `A`, `B`, `C` | nothing, but it tells you how the pixels were written (the spec sheets) |
| `scanned_at` | the scanner's clock | `2026-11-02T14:05` | other date formats, empty, `N/A` |
| `label_typed` | **the labelling team**: a person looks at this one box image and types the digit they see | `0` to `9` | typed by hand: `" 7"`, `seven`, `O` for a zero, `77`, empty, and now and then a wrong digit |
| `postcode` | **the counter clerk**: types the parcel's whole postcode to route it | `28013` | `28.013`, `E-28013`, `Madrid 28013`, a digit missing, ... |
| `box` | the scanner software: which of the five boxes of the postcode this image is | `1` to `5` | nothing |
| `pixels` | the scanner: the box image, 28 × 28 = 784 values, row by row | 784 whole numbers from 0 to 255 | it depends on the scanner; some rows are cut short or broken |
| `notes` | depot staff, free text | anything | commas, e-mail addresses, phone numbers |
| `operator_email` | the labelling operator who typed `label_typed` | `op.amber17@example.org` | personal data that the model never needs |

**Why is there a `postcode` column, if reading postcodes is the whole point?** Because a person still typed it:
the counter clerk, to route this parcel today. The labelling team typed `label_typed` separately, looking only
at the box image. So `pixels` is box number `box` of a label whose postcode is `postcode`, and `label_typed`
should be the digit in that position: `postcode[box - 1]` (minus 1, because Python counts from 0). Two people
typed them, so when they disagree, one of them made a mistake. The cell below compares them on the first rows.

In [ ]:
# ▶️ JUST RUN · the Lab 3 tools (run this once)
import csv
from labkit.scans import naive_load, run_pipeline, retrain_and_compare, as_training_set, read_rows
from labkit.lab3_tools import (ready, show_file, show_rows, show_labels, show_pixels, show_fix, show_report,
                               show_mine, save_and_show, profile_csv, show_raw_line, show_scanners,
                               show_inversion)
print("The Lab 3 tools are ready.")

In [ ]:
# ▶️ JUST RUN · postcode, box and label_typed: two people, one digit (the first 6 rows)
with open("data/parcel_scans_raw.csv", encoding="utf-8", newline="") as f:
    reader = csv.DictReader(f)
    print(f"{'scan_id':<11}{'postcode':<10}{'box':<5}{'postcode[box - 1]':<19}label_typed")
    for number, row in enumerate(reader):
        if number == 6:
            break
        box = int(row["box"])
        digit = row["postcode"][box - 1] if box <= len(row["postcode"]) else "(too short)"
        print(f"{row['scan_id']:<11}{row['postcode']:<10}{box:<5}{digit!r:<19}{row['label_typed']!r}")

The first row: `'0'` and `'Zero'` are the same digit, typed two ways (C1 deals with that). The sixth row
disagrees, but look at its postcode: the clerk typed `28.033` with a dot, so position 4 holds the wrong
character. Both columns were typed by people, and both can be messy.

### Last month vs this month

Last month's export came from scanner A only (the new scanners hadn't arrived yet), and the model trained on it
worked. `data/parcel_scans_last_month.csv` holds 20 of its rows, a small sample to compare with (not the
whole training set). The cell below shows the first lines of both files, exactly as they are in the files,
with the long pixels column shortened.

In [ ]:
# ▶️ JUST RUN · the first lines of last month's sample, then of this month's export
show_file("data/parcel_scans_last_month.csv", n=6)
print()
show_file("data/parcel_scans_raw.csv", n=8)

**What differences can you spot between last month's data and this month's?** About 3 minutes, alone or with
a neighbour: go column by column. Stuck? Open the questions one at a time.

<details>
<summary><strong>Question 1</strong></summary>

Does every `label_typed` look like one digit?

</details>

<details>
<summary><strong>Question 2</strong></summary>

Does every image have exactly 784 pixel values? (The preview counts the values of each pixels field; the first
look below counts them for the whole file.)

</details>

<details>
<summary><strong>Question 3</strong></summary>

Are all the pixel values whole numbers from 0 to 255?

</details>

<details>
<summary><strong>Question 4</strong></summary>

Why would some pixel values have decimals? Look at the scanner column, and at the spec sheets.

</details>

<details>
<summary><strong>Question 5</strong></summary>

Some images start with `255 255 255 ...`. For scanner A, 255 means ink; what does it mean for scanner B? What
does the model see?

</details>

<details>
<summary><strong>What we found, and why each one is a problem (open after you've looked)</strong></summary>

| What you see this month | Where it comes from | Why it is a problem |
|---|---|---|
| labels such as `Zero`, `" 7"`, `O` | the labelling team types freely | the model needs an `int` from 0 to 9; `int("Zero")` crashes |
| decimals such as `0.8706` | scanner C writes fractions | `int()` can't read them; the old script used `int(float(x))`, which turns every pixel into 0 or 1: a blank image |
| images that start with `255 255 255` | scanner B writes a negative | to the model, 255 is ink: it sees a black page with a white digit |
| dates in other formats, empty or `N/A` | the scanners' clocks and settings | not needed to train, but a report would show them wrong |
| `Chamartín` written several ways | the depots' systems | one depot looks like several (BLD1) |

Last month: one scanner, plain digits, 784 whole numbers in every row. Nothing in this list crashes the old
script: it reads what it can and trusts it. More problems hide further down the file; the first look below
finds them.

</details>

### A first look at the whole file: exploratory data analysis

**Exploratory data analysis (EDA)** means looking at the whole dataset through a few numbers (how many rows and
columns, how many empty values, how many different values, the range of the numbers) before you trust it or
train on it. Eight lines of a 1,500-line file can't show you everything.

`profile_csv(path)` (provided, standard library only) prints one line per column: how many cells are
**empty**, how many hold text that *means* missing (`N/A`, `unknown`, ...), how many **different** values
there are, and the most common ones; then a summary of the pixels. Run it on both files and compare them line
by line.

In [ ]:
# ▶️ JUST RUN · a first look at last month's sample and at this month's export
profile_csv("data/parcel_scans_last_month.csv")
print()
profile_csv("data/parcel_scans_raw.csv")

**How to read it:**

- **different:** last month, `label_typed` has at most 10 different values, the ten digits. How many does this
  month's have? Every extra one is another way of typing a digit, or a mistake. The same question works for
  `depot`, and for `scan_id` (fewer different ids than rows means some ids appear twice).
- **empty** and **missing-like:** to Python, `"N/A"` is ordinary text, three characters long. Only a person
  can decide that it means "missing", so `profile_csv` counts both kinds.
- **pixels:** the values per row, whole numbers or decimals, the smallest and the largest value, and how many
  images are mostly white. Compare each one with the spec sheets.

In [ ]:
# ▶️ JUST RUN · the same first look with pandas (an optional library, Theory S14: skipped if it is missing)
from labkit.optional import need
pd = need("pandas", "this optional first look")
if pd is not None:
    df = pd.read_csv("data/parcel_scans_raw.csv", dtype=str, on_bad_lines="skip", encoding="utf-8")
    df.info()
    print("\nMissing values per column, as pandas counts them:")
    print(df.isna().sum().to_string())

**What pandas decided for you.** `df.info()` lists the columns, their types and how many values are not
missing; `df.isna().sum()` counts the missing ones per column. pandas counts as missing (`NaN`) the empty
cells **and** a few spellings it knows, such as `N/A`, `n/a`, `NA` and `null`; `unknown`, `-` or `Zero` are
just text to it. `dtype=str` keeps every other value as the text in the file, as `csv.DictReader` does, and
`on_bad_lines="skip"` drops the broken lines **without saying which**: the worked example below keeps their
line numbers. (No pandas? Nothing is lost: `profile_csv` showed the same facts.)

**Which of these problems could Python catch automatically?** Think about it for a moment, then open the plan.

<details>
<summary><strong>The plan of the CORE</strong></summary>

| Problem | Caught by |
|---|---|
| a line broken by an unquoted comma | W `read_scans` (the worked example, next) |
| a label that isn't one clear digit | C1 `clean_label` |
| pixels that aren't 784 whole numbers from 0 to 255 (decimals, cut short, `NaN`, 256, -1) | C2 `parse_pixels` |
| an inverted image (scanner B) | C3 `fix_inversion` |
| the same image twice | C4 `clean_scans` (and `run_pipeline` in Putting It All Together) |
| a well-formed but wrong digit | no rule about text can see it: SCI2 (ADVANCED) uses the images |

</details>

### Incident report (about 3 minutes)

Monday morning at Garabato. Ops retrained the digit reader on the November export, exactly as every month. The
model is the **average-digit model** (Lab 1, SCI1): one average image per digit, and a new image gets the label
of the closest average. It is a very simple model, not a neural network (SCI3, ADVANCED, tries a real one).
Since the retraining, the reader makes strange mistakes.

`naive_load()` (provided) reads the export the way the old script did: it skips the rows that crash and
trusts everything else (`int(float(value))` for every pixel). `retrain_and_compare()` (provided) retrains the
model on that data and tests it on the 2,000 test images, next to **last month's model** (trained on 1,500 clean
October scans, all from scanner A). How bad is it? Run it and find out.

In [ ]:
# ▶️ JUST RUN · the incident: retrain on the export as it is, then test on 2,000 test images (a few seconds)
naive_images, naive_labels = naive_load("data/parcel_scans_raw.csv")
print(f"naive_load kept {len(naive_images):,} images of the export.")
incident = retrain_and_compare(naive=(naive_images, naive_labels))

**Why did it collapse?** The model keeps one average image per digit, and every training image goes into the
average of its label with the same weight. So what did the naive loader feed it? Here is one scan from each
scanner, exactly as `naive_load` read it.

In [ ]:
# ▶️ JUST RUN · one scan from each scanner, as the naive loader read it
export_rows, _ = read_rows("data/parcel_scans_raw.csv")      # every well-formed row, as text (W shows how)
show_scanners(export_rows)

Scanner B's negatives look like a page full of ink, and scanner C's fractions became 0s and 1s: blank images.
Together they are about half of the training images, averaged in with the good ones. The cell below draws the
average image of two digits, last month's next to this month's.

In [ ]:
# ▶️ JUST RUN · the average image of 1 and 7: last month's model next to this month's
from labkit.scans import fit_averages
if "naive_images" not in globals():       # the incident cell was skipped (or the kernel restarted)
    naive_images, naive_labels = naive_load("data/parcel_scans_raw.csv")
last_images, last_labels = load_digits("data/mnist_train_10k.csv.gz", limit=1500)    # last month's 1,500 scans
last, now = fit_averages(last_images, last_labels), fit_averages(naive_images, naive_labels)
side_by_side([[round(p) for p in last[1]], [round(p) for p in now[1]], [round(p) for p in last[7]],
              [round(p) for p in now[7]]], ["1, last month", "1, this month", "7, last month", "7, this month"])

A grey blur matches no real digit well, so the model's answers fall apart. The naive loader never crashed, and
that was the problem: it trusted every value it could read, right or wrong. The rest of the lab writes
cleaners that **check every value** and set aside what they can't trust; then *Putting It All Together*
retrains the same model on the clean rows, to see whether that fixes it.

### W · Worked example: Read the export · ▶️ JUST RUN

Before any cleaning, read the file correctly: the right encoding, every row as a dict, and the broken rows set aside with their line numbers.

**You'll practise:** `with open(...)`, `encoding`, `newline`, `csv.DictReader` and `enumerate` (Theory S5–S6)

**Signature**

```python
def read_scans(path: str) -> tuple[list[dict[str, str]], list[int]]
```

**Toy example**

```python
rows, bad_lines = read_scans("data/parcel_scans_raw.csv")
# rows: one dict per good row, e.g. rows[0]["label_typed"]
# bad_lines: the line numbers of the rows that couldn't be read cleanly
```

The code is already written: read it before you run it. First, see what the wrong encoding does to this file:
the cell below reads the same bytes twice.

In [ ]:
# ▶️ JUST RUN · the same bytes, read with two encodings (Windows' old default cp1252, and UTF-8)
from pathlib import Path
data = Path("data/parcel_scans_raw.csv").read_bytes()
start = data.find("Chamartín".encode("utf-8"))
chunk = data[start:start + 10]
print("the bytes in the file:", chunk)
print("read as cp1252:       ", chunk.decode("cp1252"))
print("read as UTF-8:        ", chunk.decode("utf-8"))

On Windows, `open()` without `encoding=` may use cp1252, an old Windows encoding: `í` (two bytes in UTF-8)
becomes `Ã­`. That is why every `open()` in this course says which encoding to use. The worked example is
already in the next cell: read it line by line, run it, then run its check. (The `;` at the end of a check
line only stops Jupyter from printing `True` or `False`.)

**How `read_scans` works, line by line:**

- `csv.DictReader(f)` reads the header line once, then gives **one dict per row**: the keys are the header's
  names, the values are the texts of that row. Always text: `row["box"]` is `"4"`, not `4`.
- `enumerate(reader, start=2)` numbers the rows as **lines of the file**: line 1 is the header, so the first data
  row is line 2. With that number you can find the line in VS Code (Go to Line: Ctrl+G).
- A row with **more values than the header** gets an extra key, `None`, holding the values that had no column
  left. `None in row` finds those rows.
- `rows` collects the good dicts; `bad_lines` collects the line numbers of the broken ones, so they can be
  reported instead of guessed.

In [ ]:
# ▶️ JUST RUN · the worked example: already written, so read it line by line, then run it
def read_scans(path: str) -> tuple[list[dict[str, str]], list[int]]:
    """Read a scanner export: return (good rows, line numbers of malformed rows).

    A malformed row has MORE values than the header, because a comma inside a note was not quoted:
    csv.DictReader puts the extra values under the key None.
    """
    rows = []
    bad_lines = []
    with open(path, encoding="utf-8", newline="") as f:   # the file's encoding; csv handles line ends itself
        reader = csv.DictReader(f)                         # one dict per row, keys from the header line
        for line_number, row in enumerate(reader, start=2):   # line 1 is the header
            if None in row:                                # an unquoted comma: an extra value, key None
                bad_lines.append(line_number)
            else:
                rows.append(row)
    return rows, bad_lines

In [ ]:
# ▶️ JUST RUN · check your work
check("W", read_scans);

In [ ]:
# ▶️ JUST RUN · read the export with the worked example, then look at its first malformed line three ways
rows, bad_lines = read_scans("data/parcel_scans_raw.csv")
print(f"{len(rows):,} good rows; malformed rows on lines {bad_lines}")
print()
show_raw_line("data/parcel_scans_raw.csv", bad_lines[0])

**From the file to a broken dict.** In the file, the note `label smudged, re-scanned` has a comma and no quotes
around it. The csv module splits at every comma that isn't inside quotes, so the note became two values, every
value after it moved one column along, and the e-mail address had no column left: `DictReader` put it under the
key `None`. The same note **in quotes** is fine: look at line 6 of the export in the preview above, where the
csv module reads the comma as part of the value. (If you read the file with `lines = f.read().splitlines()`, this line is `lines[142]`: a list counts from
0, a file from line 1.) We can't trust which value belongs to which column, so `read_scans` sets the row aside.

**Well-formed is not the same as correct.** `read_scans` checks only the *shape* of a row: one value per column.
The very first good row has `label_typed` `'Zero'`, and other good rows hold 783 pixels, decimals or an inverted
image. Checking the *values* is the job of the CORE.

## CORE: the tasks everyone does

**C1 → C2 → C3 → Putting It All Together → C4.** The first four are the **minimum path**: reaching Putting It All Together is a complete lab. C4 completes the CORE.

For each task: read the card, write your code in the cell that starts with `# ===== YOUR CODE HERE =====` (replace `pass`), run that cell, then run the check under it. The check shows the tests it ran: what each one checks, its inputs, the expected result and your result.

### C1 · Which digit did they type? · ✏️ WRITE CODE

The labelling team typed every label by hand, looking at one box image: ' 7', 'seven', 'O' for a zero, '77'. Turn the text into the digit it means, or None when you can't trust it.

**You'll practise:** string methods, dict lookup, and None for a value you can't trust (Theory S6) · **Typical time:** 8–12 min

**Minimum path: you need this for Putting It All Together.**

**Signature**

```python
def clean_label(raw: str) -> int | None
```

**Toy example**

```python
clean_label(" 7")       # 7     (strip the spaces)
clean_label("SEVEN")    # 7     (lower case, then LABEL_WORDS)
clean_label("O")        # 0     (a look-alike letter: LOOK_ALIKES)
clean_label("77")       # None  (two digits: not a label)
```

**What C1 does, and what it doesn't.** The labels are manual transcriptions, so they carry the usual typing
habits: spaces around the digit, a number written as a word (`seven`, `Seven`), a letter that looks like a digit
(`O` for 0, `l` or `I` for 1), and plain slips (`77`, `7?`, an empty value). C1 **normalises** what was typed;
it doesn't look at the image again (recognising digits is the model's job). And it doesn't force every text
into a digit: when you can't be sure what the person meant, return `None`, and the row is set aside with its
reason.

**Why not just `int(raw)`?** Because `int()` alone gets it wrong both ways:

| Typed | `int(typed)` | What we want |
|---|---|---|
| `" 7"` | `7` | `7` |
| `"seven"` | `ValueError` (the pipeline stops) | `7` |
| `"7.0"` | `ValueError` (the pipeline stops) | `None` |
| `"77"` | `77` (a wrong label, silently) | `None` |
| `"00"` | `0` (two characters read as one digit, silently) | `None` |

So look at the text first: **normalise** it (`strip`, `lower`), **map** it (one digit, a word, a look-alike),
and return `None` for everything else.

**Stuck for 7 minutes after Hint 3?** Run `clean_label = rescue("C1")` in a new cell, carry on, and come back to C1 later.

In [ ]:
# ▶️ JUST RUN · the two dicts clean_label uses (provided: don't modify them)
LABEL_WORDS = {"zero": 0, "one": 1, "two": 2, "three": 3, "four": 4,
               "five": 5, "six": 6, "seven": 7, "eight": 8, "nine": 9}
LOOK_ALIKES = {"o": 0, "l": 1, "i": 1}      # the letter o, a lower-case L, a capital i (after .lower())

In [ ]:
# ===== YOUR CODE HERE =====
# ✏️ WRITE CODE: replace `pass` with your code, then run this cell (Shift+Enter).
def clean_label(raw: str) -> int | None:
    """Return the digit a person typed (an int from 0 to 9), or None if you can't trust it.

    >>> clean_label(" Seven ")
    7
    """
    # TODO 1: normalise: no spaces around the text, and lower case
    # TODO 2: one digit, "0" to "9"? return it as an int
    # TODO 3: a word in LABEL_WORDS? return its digit. A letter in LOOK_ALIKES? return its digit
    # TODO 4: anything else: return None
    pass

In [ ]:
# ▶️ JUST RUN · check your work
check("C1", clean_label);

In [ ]:
# ▶️ JUST RUN · the messiest labels of the export, read by YOUR clean_label
show_labels(clean_label, rows)

<details>
<summary><strong>Hint 1 · Nudge</strong></summary>

Normalise, then map, then give up: clean the text first (Theory S6, string methods), then try each way a label can be written, and return `None` at the end for everything else.

</details>

<details>
<summary><strong>Hint 2 · Plan</strong></summary>

1. `text = raw.strip().lower()`.
2. If `len(text) == 1 and text in "0123456789"`: return `int(text)`.
3. If `text in LABEL_WORDS`: return `LABEL_WORDS[text]`. If `text in LOOK_ALIKES`: return `LOOK_ALIKES[text]`.
4. After the ifs (not indented under them): `return None`.

</details>

<details>
<summary><strong>Hint 3 · Skeleton</strong></summary>

    text = raw.___().___()
    if len(text) == ___ and text in "0123456789":
        return int(___)
    if text in LABEL_WORDS:
        return LABEL_WORDS[___]
    if text in ___:
        return LOOK_ALIKES[text]
    return ___

</details>

<details>
<summary><strong>Hint 4 · Ask an AI tutor (not for the answer)</strong></summary>

Copy this into your AI assistant: *"I'm learning Python. Act as a tutor and do NOT write code for me. Here is my function and the check output: [paste]. Ask me one question at a time that helps me find the problem myself."*

</details>

### C2 · 784 numbers, or nothing · ✏️ WRITE CODE

Each box image arrives as one long text of numbers separated by spaces: 784 whole numbers from scanner A, fractions such as 0.4627 from scanner C, and now and then a row cut short or broken. Turn it into 784 ints from 0 to 255, or None if anything is wrong.

**You'll practise:** `split`, `int()` inside `try/except`, and checking length and range (Theory S2, Theory S6) · **Typical time:** 8–12 min

**Minimum path: you need this for Putting It All Together.**

**Signature**

```python
def parse_pixels(raw: str) -> list[int] | None
```

**Toy example**

```python
parse_pixels("0 0 255 ... 0")         # 784 values  -> [0, 0, 255, ..., 0]
parse_pixels("0 0 255 ... 0")         # 783 values  -> None
parse_pixels("0 NaN 255 ... 0")       # int("NaN") raises ValueError -> None
parse_pixels("0.0 0.4627 ... 0.0")    # decimals (scanner C) -> None, for now
```

**Why `try/except` around `int()`, instead of checking each piece first?** `int()` is the authority on what
a whole number is; asking it directly avoids guessing:

| Piece | `piece.isdigit()` | `int(piece)` | Then |
|---|---|---|---|
| `"255"` | True | `255` | in range: a pixel |
| `"-1"` | False | `-1` | out of range: `None` |
| `"NaN"` | False | `ValueError` | the `except` block: `None` |
| `"0.4627"` | False | `ValueError` | the `except` block: `None` |

`try/except ValueError` catches exactly "this isn't a whole number"; the range test then decides what a
pixel is. Two separate questions, two separate tests.

**`int("7")` is 7, but `int("7.0")` raises `ValueError`.** On a text, `int()` accepts whole numbers only (spaces
around them are fine). That is exactly why every scanner C row fails: its pixels are fractions. In the CORE we
**reject** them (`None`): a rule we can state and check. SCI1 (OPTIONAL) rescues them later, once rejecting works.

**Why `except ValueError`, and not a bare `except:`?** `ValueError` means exactly "this text isn't a whole
number". A bare `except:` (or `except Exception:`) would also swallow your own mistakes, such as a misspelt
variable (`NameError`) or a wrong type (`TypeError`): the function would quietly return `None` for every row,
and the bug would look like bad data.

**Stuck for 7 minutes after Hint 3?** Run `parse_pixels = rescue("C2")` in a new cell, carry on, and come back to C2 later.

In [ ]:
# ===== YOUR CODE HERE =====
# ✏️ WRITE CODE: replace `pass` with your code, then run this cell (Shift+Enter).
def parse_pixels(raw: str) -> list[int] | None:
    """Return the 784 pixels as ints from 0 to 255, or None if anything is wrong."""
    # TODO 1: split the text on whitespace; not exactly 784 pieces? return None
    # TODO 2: convert each piece with int() inside try/except ValueError (except: return None)
    # TODO 3: a value below 0 or above 255? return None
    # TODO 4: after the loop, return the list of ints
    pass

In [ ]:
# ▶️ JUST RUN · check your work
check("C2", parse_pixels);

In [ ]:
# ▶️ JUST RUN · five kinds of pixel text from the export, read by YOUR parse_pixels
show_pixels(parse_pixels, rows)

<details>
<summary><strong>Hint 1 · Nudge</strong></summary>

One loop over the pieces of `raw.split()`. Put `int(piece)` inside a `try` block: its `except ValueError` block is your exit for text that isn't a whole number (Theory S6, a value or None).

</details>

<details>
<summary><strong>Hint 2 · Plan</strong></summary>

1. `pieces = raw.split()` (no argument: any amount of whitespace). If `len(pieces) != 784`, return `None`.
2. Start `pixels = []`. For each `piece` in `pieces` (or `for i in range(len(pieces)):` with `pieces[i]`): `try: value = int(piece)` and `except ValueError: return None`.
3. If `value < 0 or value > 255`, return `None`; otherwise `pixels.append(value)`.
4. After the loop, return `pixels`.

</details>

<details>
<summary><strong>Hint 3 · Skeleton</strong></summary>

    pieces = raw.split()
    if len(pieces) != ___:
        return None
    pixels = []
    for piece in pieces:
        try:
            value = ___(piece)
        except ___:
            return None
        if value < ___ or value > ___:
            return None
        pixels.append(___)
    return ___

</details>

<details>
<summary><strong>Hint 4 · Ask an AI tutor (not for the answer)</strong></summary>

Copy this into your AI assistant: *"I'm learning Python. Act as a tutor and do NOT write code for me. Here is my function and the check output: [paste]. Ask me one question at a time that helps me find the problem myself."*

</details>

### C3 · White paper, black ink? · ✏️ WRITE CODE

Scanner B writes paper as 255 and ink as 0: a negative. Find those images and turn them back, without touching the others.

**You'll practise:** a mean, a condition and a list comprehension (Theory S2, Theory S4) · **Typical time:** 4–6 min

**Minimum path: you need this for Putting It All Together.**

**Signature**

```python
def fix_inversion(pixels: list[int]) -> list[int]
```

**Toy example**

```python
fix_inversion([0, 0, 255, 10])       # mean 66: a normal scan     -> [0, 0, 255, 10] (a new list)
fix_inversion([255, 255, 0, 245])    # mean 189: white paper      -> [0, 0, 255, 10]
```

**First, the goal.** The cell below shows a scanner B scan as it arrives, the same scan with every pixel
turned into `255 - pixel`, and a scanner A scan for comparison, each with its mean pixel.

**Stuck for 7 minutes after Hint 3?** Run `fix_inversion = rescue("C3")` in a new cell, carry on, and come back to C3 later.

In [ ]:
# ▶️ JUST RUN · C3's goal, before any code: a scanner B scan, the same scan as 255 - pixel, a scanner A scan
show_inversion(rows)

**How can a function tell that a scan is inverted?** A digit covers a small part of the box, so most pixels of
any scan are paper. On a normal scan paper is 0, so the **mean pixel** is low; on a scanner B scan paper is 255,
so the mean is high. **127** is halfway between 0 and 255: above it, the image is mostly white paper. The cell
below prints the mean pixel of a few real scans of scanners A and B: does a line at 127 separate them?

**How do we turn it back?** `255 - pixel` swaps the two ends of the scale: 255 (paper on B) becomes 0, 0 (ink
on B) becomes 255, and a grey 200 becomes 55. Do it for every pixel, in a new list.

**A rule of thumb, not a law.** The mean is a *heuristic*: it works on these scanners, but a very thick digit, or
a dark smudge on the paper, could push a normal scan above 127. Here the scanner column would also tell us, but a
rule that looks at the image itself also works for a scanner nobody told us about.

**Why a new list, even when nothing changes?** If `fix_inversion` returned the list it was given, the caller's
list and the result would be the same object: changing one would change the other. A copy keeps them apart.

In [ ]:
# ▶️ JUST RUN · the mean pixel of the first good scans of scanners A and B
for scanner in ("A", "B"):
    found = [row for row in rows if row["scanner"] == scanner and row["pixels"].replace(" ", "").isdigit()][:6]
    means = [sum(int(v) for v in row["pixels"].split()) / len(row["pixels"].split()) for row in found]
    print(f"scanner {scanner}:", "  ".join(f"{mean:.0f}" for mean in means))

In [ ]:
# ===== YOUR CODE HERE =====
# ✏️ WRITE CODE: replace `pass` with your code, then run this cell (Shift+Enter).
def fix_inversion(pixels: list[int]) -> list[int]:
    """Return the image with paper 0 and ink 255: if the mean pixel is above 127, every pixel becomes
    255 - pixel. Always return a NEW list.

    >>> fix_inversion([255, 255, 0, 245])
    [0, 0, 255, 10]
    """
    # TODO 1: the mean pixel: sum divided by how many
    # TODO 2: above 127? return a new list with 255 - p for every pixel p
    # TODO 3: otherwise return a copy of pixels
    pass

In [ ]:
# ▶️ JUST RUN · check your work
check("C3", fix_inversion);

In [ ]:
# ▶️ JUST RUN · a scanner B scan before and after YOUR fix_inversion, and a scanner A scan (unchanged)
show_fix(fix_inversion, rows)

<details>
<summary><strong>Hint 1 · Nudge</strong></summary>

Two steps: compute the mean (the sum divided by how many), then decide. A list comprehension builds the inverted list in one line (Theory S4); `list(pixels)` makes a copy.

</details>

<details>
<summary><strong>Hint 2 · Plan</strong></summary>

1. `mean = sum(pixels) / len(pixels)` (a single `/`, so 127.5 stays 127.5).
2. If `mean > 127`: return `[255 - p for p in pixels]` (or build it with a loop and `append`).
3. Otherwise return a copy: `list(pixels)`, never `pixels` itself.

</details>

<details>
<summary><strong>Hint 3 · Skeleton</strong></summary>

    mean = sum(___) / len(___)
    if mean > ___:
        return [___ - p for p in pixels]
    return list(___)

</details>

<details>
<summary><strong>Hint 4 · Ask an AI tutor (not for the answer)</strong></summary>

Copy this into your AI assistant: *"I'm learning Python. Act as a tutor and do NOT write code for me. Here is my function and the check output: [paste]. Ask me one question at a time that helps me find the problem myself."*

</details>

### Putting It All Together · ▶️ JUST RUN

This is where your three cleaners work together, on the whole export. When you run the cell below:

1. `run_pipeline()` (provided) **cleans every row** with a complete cleaning pipeline written for you, with
   YOUR three functions plugged in: `clean_label` (C1) reads the label, `parse_pixels` (C2) reads the pixels,
   `fix_inversion` (C3) turns scanner B's images the right way round;
2. it adds **one rule of its own**: an image already seen (the same `scan_id`) is rejected as a duplicate;
3. it **keeps the good rows** and records why every other row was rejected: `show_report()` prints that report;
4. `retrain_and_compare()` **retrains the same average-digit model** on your clean rows,
5. and **tests it** on the same 2,000 test images as the incident, next to last month's model and the naive data.

Did cleaning fix the model? Run it and find out. (In C4, right after, you write that pipeline yourself.)

In [ ]:
# ▶️ JUST RUN · Putting It All Together: clean the export with YOUR three cleaners, retrain, compare (a few seconds)
clean_rows, report = run_pipeline(rows, clean_label, parse_pixels, fix_inversion)
if clean_rows:
    show_report(report, clean_rows)
    if "naive_images" not in globals():       # the incident cell was skipped (or the kernel restarted)
        naive_images, naive_labels = naive_load("data/parcel_scans_raw.csv")
    results = retrain_and_compare(naive=(naive_images, naive_labels), cleaned=clean_rows)

**Reaching Putting It All Together is a complete lab.** Think about it (the debrief comes back to these):

- Which rule rejected the most rows? Did that surprise you?
- Which defect do you think hurt the model most: inverted images, decimals, duplicates, wrong labels?
- The clean data has fewer images than the naive data. Why is the model better anyway?

Next: C4, where you write the pipeline you just used; then *Check yourself*; then regular expressions and an AI
red team; then the OPTIONAL tasks (C5 saves your clean data and its report) and the ADVANCED ones.

### C4 · One pipeline, rules in order · ✏️ WRITE CODE

Putting It All Together ran a provided pipeline. Its rules are decisions (a copy is a copy; a label you can't read is not a label), and someone has to own them: write them yourself, in a fixed order you can read top to bottom, keep only what training needs, and return a report the scanner vendors can act on.

**You'll practise:** composing your cleaners, a set for "seen before?" and a report dict (Theory S3, Theory S5) · **Typical time:** 12–18 min

**Signature**

```python
def clean_scans(rows: list[dict[str, str]]) -> tuple[list[dict], dict]
```

**Toy example**

```python
clean_rows, report = clean_scans(rows)
clean_rows[0]    # {'scan_id': 'PX-001001', 'depot': 'Retiro', 'label': 0, 'pixels': [0, 0, ...]}
report           # {'total': ..., 'valid': ..., 'invalid': ..., 'reasons': {'duplicate': ..., ...}}
```

**In Putting It All Together you used a provided pipeline, `run_pipeline`. Now you write the same logic
yourself**, so that every rule is yours: you can read it, explain it, and change it.

**The rules, in this order** (a row is rejected for the FIRST rule it breaks, so every rejected row has exactly
one reason):

| Order | Test | Reason recorded |
|---|---|---|
| 1 | the id `scan_id.strip().upper()` is already in `seen` (otherwise add it, right away) | `"duplicate"` |
| 2 | `clean_label(label_typed)` is `None` | `"bad label"` |
| 3 | `parse_pixels(pixels)` is `None` | `"bad pixels"` |
| 4 | none of the above: keep `{"scan_id", "depot" (stripped), "label", "pixels": fix_inversion(...)}` | |

The report: `{"total": rows read, "valid": rows kept, "invalid": rows rejected, "reasons": {reason: count}}`
(a reason that never happened may be left out).

**Why this order?**
- **Duplicates first:** a second copy is the same image again, whatever its label or pixels say. If the label
  came first, a copy of a row with a bad label would count as a second "bad label": one problem, reported
  twice. So the id goes into `seen` as soon as it passes this test, and every later copy is a `"duplicate"`.
- **Label, then pixels:** a row needs both, so either order keeps the same rows. The label test reads one short
  text, the pixels test reads 784 values: the cheap test goes first. And a fixed order means the same data
  always gives the same report.
- **One reason per row:** the report goes to the scanner vendors, and they must be able to act on it. If a row
  could count under two reasons, the reasons would add up to more rows than were rejected.

**Why keep only four columns?** *Data minimisation* (GDPR): keep only what the purpose needs. Training needs
the image and its label; `scan_id` and `depot` let us trace a row back. `operator_email` identifies an employee,
and `notes` sometimes holds a customer's phone number or e-mail address: personal data the model never needs,
so they stay behind. (`scanned_at`, `postcode`, `box` and `scanner` aren't personal, but training doesn't need
them either.)

**Stuck for 7 minutes after Hint 3?** Run `clean_scans = rescue("C4")` in a new cell, carry on, and come back to C4 later.

In [ ]:
# ===== YOUR CODE HERE =====
# ✏️ WRITE CODE: replace `pass` with your code, then run this cell (Shift+Enter).
def clean_scans(rows: list[dict[str, str]]) -> tuple[list[dict], dict]:
    """Clean every row with the three rules, in order; return (clean_rows, report)."""
    # TODO 1: before the loop: an empty list of clean rows, an empty dict of reasons, an empty set `seen`
    # TODO 2: for each row, test the rules in the order of the table (if / elif / else);
    #         the first rule that fails gives the reason; add the id to seen right after rule 1
    # TODO 3: a row that passes every rule: append a NEW dict with scan_id, depot, label, pixels
    # TODO 4: count each reason in the dict; after the loop build the report and return both
    pass

In [ ]:
# ▶️ JUST RUN · check your work
check("C4", clean_scans);   # toy rows, then the 30-row fixture, then the whole export

The check works in three steps: **toy rows** (one rule at a time), then the **30-row fixture**
(`data/fixtures/scans_fixture.csv`, every kind of defect: the first row your function gets wrong is
shown), then the **whole export**, reason by reason.

In [ ]:
# ▶️ JUST RUN · YOUR clean_scans on the whole export, compared with the expected numbers and with run_pipeline
mine = show_mine(clean_scans, rows, report)

<details>
<summary><strong>Hint 1 · Nudge</strong></summary>

One loop over the rows and one `if / elif / else` chain in the order of the table: the first test that fails gives the reason. A set remembers the ids you've seen; a dict counts the reasons (Theory S3).

</details>

<details>
<summary><strong>Hint 2 · Plan</strong></summary>

1. Before the loop: `clean_rows = []`, `reasons = {}`, `seen = set()`.
2. For each row: `reason = None` and `scan_id = row["scan_id"].strip().upper()`. Then: if `scan_id in seen`, the reason is `"duplicate"`; else `seen.add(scan_id)`, then test `clean_label(row["label_typed"]) is None` (`"bad label"`) and `parse_pixels(row["pixels"]) is None` (`"bad pixels"`). If both are fine, append a NEW dict with the four keys, the pixels through `fix_inversion`.
3. If `reason is not None`: `reasons[reason] = reasons.get(reason, 0) + 1`.
4. After the loop: `report = {"total": len(rows), "valid": len(clean_rows), "invalid": len(rows) - len(clean_rows), "reasons": reasons}`; return `clean_rows, report`.

</details>

<details>
<summary><strong>Hint 3 · Skeleton</strong></summary>

    clean_rows, reasons, seen = [], {}, set()
    for row in rows:
        reason = None
        scan_id = row["scan_id"].strip().___()
        if scan_id in ___:
            reason = "duplicate"
        else:
            seen.___(scan_id)
            label = clean_label(row["label_typed"])
            pixels = parse_pixels(row["pixels"])
            if label is None:
                reason = ___
            elif pixels is None:
                reason = ___
            else:
                clean_rows.append({"scan_id": scan_id, "depot": row["depot"].strip(),
                                   "label": label, "pixels": ___(pixels)})
        if reason is not None:
            reasons[reason] = reasons.get(reason, 0) + 1
    report = {"total": ___, "valid": ___, "invalid": ___, "reasons": reasons}
    return clean_rows, report

</details>

<details>
<summary><strong>Hint 4 · Ask an AI tutor (not for the answer)</strong></summary>

Copy this into your AI assistant: *"I'm learning Python. Act as a tutor and do NOT write code for me. Here is my function and the check output: [paste]. Ask me one question at a time that helps me find the problem myself."*

</details>

**CORE complete.** Well done. Next, on your own: *Check yourself*, four exam-style questions on today's
Python. Then regular expressions and an AI red team; then any OPTIONAL task (C5 saves your clean data and
its report), or ADVANCED if you move faster.

## Check yourself (5 minutes, exam-style)

Exam-style questions on today's ideas. Answer on paper or in your head first, then open the answer: nothing to run, nothing is checked.

**1. What does this print? (C1 tests `len(text) == 1` for a reason.)**

```python
digits = "0123456789"
for text in ["7", "77", "56", ""]:
    print(repr(text), text in digits)
```

<details>
<summary><strong>Answer</strong></summary>

`'7' True`, `'77' False`, `'56' True`, `'' True`. Between two strings, `in` asks "is it a piece of the text?", not "is it one of the characters?": `"56"` is a piece of `"0123456789"`, and the empty string is a piece of every string. That's why `clean_label` also tests `len(text) == 1` (C1 has a test for it: `"12"`).

</details>

**2. What is wrong here? It never crashes.**

```python
def parse_pixels(raw):
    pixels = []
    for piece in raw.split():
        try:
            pixels.append(int(piece))
        except ValueError:
            pass
    return pixels
```

<details>
<summary><strong>Answer</strong></summary>

`except ValueError: pass` hides the problem: `"0 NaN 255 ..."` gives a list with one value missing, and the image is silently shifted. Nothing checks for 784 values or the range 0 to 255 either. A cleaner that can't trust its input returns `None`, so the pipeline rejects the row and records why.

</details>

**3. Which structure would you use, and why? (a) the scan ids you have already seen; (b) how many rows each reason rejected; (c) the clean rows, in the order of the file.**

<details>
<summary><strong>Answer</strong></summary>

(a) a **set**: `scan_id in seen` answers instantly, and each id is stored once. (b) a **dict** `{reason: count}`, with `reasons[r] = reasons.get(r, 0) + 1`. (c) a **list** of dicts: it keeps the order, and each row keeps its column names.

</details>

**4. What is wrong here? It prints 0, although the export has many rows with the label 7.**

```python
import csv
with open("data/parcel_scans_raw.csv", encoding="utf-8", newline="") as f:
    rows = list(csv.DictReader(f))
sevens = 0
for row in rows:
    if row["label_typed"] == 7:
        sevens += 1
print(sevens)
```

<details>
<summary><strong>Answer</strong></summary>

Every value `csv.DictReader` gives you is a `str` (a CSV file is text), and `"7" == 7` is `False`: no error, just a wrong answer. Convert first, `clean_label(row["label_typed"]) == 7`, or compare text with text, `row["label_typed"] == "7"` (which still misses `" 7"` and `seven`: that is why C1 exists).

</details>

## After the CORE: regular expressions and an AI red team

About 7–10 minutes, at your own pace, whenever your CORE is done (or whenever you want a change of task).
Four short steps: the pieces of a pattern, a few experiments, a pattern of your own for Madrid postcodes,
and then an AI's pattern, put to the test.

### Step 1 · Four pieces of a regular expression · ▶️ JUST RUN

A **regular expression** (a *regex*) is a small pattern language for **finding** text inside other text
(Theory S6). `re.search(pattern, text)` looks for the pattern anywhere in the text and returns a *match*
(`match.group()` is the text it found), or `None` when there is nothing to find. Write patterns as raw
strings, `r"..."`, so that Python keeps every backslash as it is.

| Piece | Means | For example |
|---|---|---|
| `\d` | one digit, 0 to 9 | `\d` finds the `7` in `box 7` |
| `{3}` | the piece before it, three times | `\d{3}` finds three digits in a row |
| `\b` | a word boundary: the edge between a letter, digit or `_` and anything else | `\b\d{5}\b` finds five digits that stand on their own |
| `28` | a plain character matches itself | `28\d{3}` finds 28 followed by three digits |

In [ ]:
# ▶️ JUST RUN · five searches: what does each pattern find?
import re
for pattern, text in [(r"\d", "box 7 of 5"), (r"\d{3}", "PX-001234"), (r"\d{5}", "Retiro 28009, gate 4"),
                      (r"\b\d{5}\b", "order 4528013"), (r"\d{5}", "no digits here")]:
    match = re.search(pattern, text)
    print(f"re.search(r\"{pattern}\", {text!r})".ljust(46), "->", repr(match.group()) if match else None)

### Step 2 · Experiment · ✏️ WRITE CODE

Change the pattern in the cell below, run it, and watch which texts it finds something in. Try `\d{5}`,
then `28\d{3}`, then `\b28\d{3}\b`. Which texts does each one get wrong?

In [ ]:
# ✏️ WRITE CODE · change the pattern, run the cell, and compare
import re
texts = ["Madrid 28013", "(28009)", "order 4528013", "28099", "E-28013", "Madrid28013", "call 07700 900461",
         "PX-028013"]
pattern = r"\d{5}"                  # change it, then run the cell again
for text in texts:
    match = re.search(pattern, text)
    print(f"{text!r:22} ->", repr(match.group()) if match else None)

### Step 3 · Build: a Madrid postcode inside messy text · ✏️ WRITE CODE

Garabato delivers to **28001 to 28055** only. Five digits are not enough: `\d{5}` also finds `07700`
in a phone number, `28099` (not a postcode Garabato delivers to) and a piece of a longer number. Finding
five digits is not the same as finding a valid postcode.

`gauntlet("postcode", pattern)` tries a pattern on 20 tricky cases. A case passes when the pattern finds
exactly the postcode (nothing more, nothing less), or finds nothing when there is no postcode to find.
Improve your pattern case by case: all 20 is a stretch goal, which takes the pieces in the box below.

> **New tool: more regex pieces**
>
> | Piece | Means |
> |---|---|
> | `[0-4]` | one character from 0 to 4 |
> | `a\|b` | a or b; `(?:...)` groups the pieces without remembering them |
> | `(...)` | a group the match remembers: `match.group(1)` is its text (the gauntlet accepts it too) |
> | `^` and `$` | the start and the end of the text |
> | `(?<!\d)` | "not right after a digit"; `(?!\d)` "not right before a digit" |
>
> `re.search(pattern, text)` finds the first match anywhere; `re.fullmatch(pattern, text)` asks whether the
> WHOLE text matches.

In [ ]:
# ===== YOUR CODE HERE =====
# ✏️ WRITE CODE · your pattern for Madrid postcodes: improve it case by case (all 20 is a stretch goal)
from labkit.gauntlet import gauntlet
my_pattern = r"\d{5}"          # improve it
result = gauntlet("postcode", my_pattern)

### Step 4 · AI red team · ✏️ WRITE CODE

AI assistants write a regex in seconds. Is it right? Put it through the same 20 cases.

1. **Ask an AI assistant:** *"Write a Python regular expression that finds a Madrid postcode (28001 to
   28055) inside messy text."* (This prompt has no personal data in it; keep it that way.)
2. **Paste its pattern** into the next cell, between the quotes of `pattern = r"..."`, and run the cell.
3. **Read the FAIL lines.** Which cases did the AI get wrong, and why? Compare them with your pattern from
   step 3. Ask the AI to fix one failure, or fix it yourself, and run again.

No AI assistant at hand? Run the cell as it is: it holds a typical first answer.

In [ ]:
# ✏️ WRITE CODE · paste the AI's pattern between the quotes (keep the r in front), then run this cell
pattern = r"28\d{3}"            # a typical first answer: replace it with the AI's pattern
result = gauntlet("postcode", pattern)

<details>
<summary><strong>What we found (open after you've tried)</strong></summary>

The first answers usually look like `28\d{3}` or `\b28\d{3}\b`. The first finds `28013` inside longer
numbers (`128013`, `280130`) and accepts `28000` and `28099`, which are not postcodes Garabato delivers to.
Word boundaries `\b` fix the longer numbers, but fail on `Madrid28013` and `28013_retiro`: a letter, a
digit and `_` all count as word characters. A pattern that passes all 20 uses lookarounds and spells out
the range, as in step 3. The lesson: an AI answer is a draft; test it on the cases that matter
to you before you trust it.

</details>

## OPTIONAL: extra practice

Extra practice for when your CORE is done. Pick any task, in any order: each one is independent, has a `check()` and hints, and nothing later depends on it.

| Task | What you'll learn | Difficulty | Typical time |
|---|---|---|---|
| **C5** Save the clean data and the report | the clean rows as CSV and the quality report as JSON: `csv.DictWriter`, `json.dump`, `pathlib` | gentle | 6–8 min |
| **ENG1** Postcodes | a cleaner with string methods and a set lookup | gentle | 8–10 min |
| **SCI1** Rescue scanner C | turn decimals into pixels, and measure which model suffers from which defect | moderate | 12–15 min |
| **BLD1** Depot names | Unicode: two ways to write í, mojibake, cp1252 and casefold; then a table per depot | moderate | 12–15 min |

In [ ]:
# ▶️ JUST RUN · tools for the OPTIONAL and ADVANCED tasks (run this once)
from labkit import load_valid_postcodes
from labkit.gauntlet import gauntlet
from labkit.lab3_tools import depot_table, find_suspects, show_suspects
from labkit.scans import model_table

### OPTIONAL · C5 · Save the clean data and the report · ✏️ WRITE CODE

Ops want two files: the clean scans as CSV for next month's retraining, and the quality report as JSON for the scanner vendors.

**What you'll learn:** the clean rows as CSV and the quality report as JSON: `csv.DictWriter`, `json.dump`, `pathlib` (Theory S5) · **Difficulty:** gentle · **Typical time:** 6–8 min

**Signature**

```python
def save_outputs(clean_rows: list[dict], report: dict, out_dir: str = "output") -> None
```

**Toy example**

```python
save_outputs(clean_rows, report)
# output/scans_clean.csv       scan_id,depot,label,pixels   (pixels: 784 numbers joined by spaces)
# output/quality_report.json   the report, indented, accents kept as they are
```

**Why these arguments?** Each one prevents a real problem:

| Argument | Without it |
|---|---|
| `encoding="utf-8"` | Windows may write `Chamartín` as cp1252 bytes, and the next program reads mojibake |
| `newline=""` | on Windows an empty line appears after every row (the csv module writes its own line ends) |
| `indent=2` | the whole report on one long line (fine for a program, tiring for a person) |
| `ensure_ascii=False` | the JSON says `Chamart\u00edn` instead of `Chamartín` |

**Stuck for 7 minutes after Hint 3?** Run `save_outputs = rescue("C5")` in a new cell, carry on, and come back to C5 later.

In [ ]:
# ===== YOUR CODE HERE =====
# ✏️ WRITE CODE: replace `pass` with your code, then run this cell (Shift+Enter).
import csv
import json
from pathlib import Path


def save_outputs(clean_rows: list[dict], report: dict, out_dir: str = "output") -> None:
    """Write out_dir/scans_clean.csv and out_dir/quality_report.json; create out_dir if needed."""
    # TODO 1: create the folder out_dir (and don't fail if it is already there)
    # TODO 2: scans_clean.csv: a DictWriter with the 4 columns; the pixels joined by spaces
    # TODO 3: quality_report.json: json.dump the report, indented, with accents kept readable
    pass

In [ ]:
# ▶️ JUST RUN · check your work
check("C5", save_outputs);

In [ ]:
# ▶️ JUST RUN · save the clean rows and report of Putting It All Together with YOUR save_outputs, then look at the files
save_and_show(save_outputs, clean_rows, report, "output")

<details>
<summary><strong>Hint 1 · Nudge</strong></summary>

Two `with open(...)` blocks, one per file, inside a folder you create first. `csv.DictWriter` writes dicts; `json.dump` writes the report (Theory S5).

</details>

<details>
<summary><strong>Hint 2 · Plan</strong></summary>

1. `folder = Path(out_dir)` and `folder.mkdir(parents=True, exist_ok=True)`.
2. `with open(folder / "scans_clean.csv", "w", encoding="utf-8", newline="") as f:` make `writer = csv.DictWriter(f, fieldnames=["scan_id", "depot", "label", "pixels"])`, call `writer.writeheader()`, then for each row `writer.writerow({...})` with `"pixels": " ".join(str(p) for p in row["pixels"])`.
3. `with open(folder / "quality_report.json", "w", encoding="utf-8") as f:` then `json.dump(report, f, indent=2, ensure_ascii=False)`.

</details>

<details>
<summary><strong>Hint 3 · Skeleton</strong></summary>

    folder = Path(out_dir)
    folder.mkdir(parents=True, exist_ok=___)
    with open(folder / "scans_clean.csv", "w", encoding=___, newline=___) as f:
        writer = csv.DictWriter(f, fieldnames=["scan_id", "depot", "label", "pixels"])
        writer.___()
        for row in clean_rows:
            writer.writerow({"scan_id": row["scan_id"], "depot": row["depot"], "label": row["label"],
                             "pixels": " ".join(___(p) for p in row["pixels"])})
    with open(folder / "quality_report.json", "w", encoding="utf-8") as f:
        json.dump(report, f, indent=___, ensure_ascii=___)

</details>

<details>
<summary><strong>Hint 4 · Ask an AI tutor (not for the answer)</strong></summary>

Copy this into your AI assistant: *"I'm learning Python. Act as a tutor and do NOT write code for me. Here is my function and the check output: [paste]. Ask me one question at a time that helps me find the problem myself."*

</details>

### OPTIONAL · ENG1 · Postcodes · Engineer track · ✏️ WRITE CODE

The counter clerks type every postcode by hand, and Ops route parcels by it. Clean the postcode column: one canonical form, or None for a postcode Garabato doesn't deliver to.

**What you'll learn:** a cleaner with string methods and a set lookup (Theory S3, Theory S6) · **Difficulty:** gentle · **Typical time:** 8–10 min

**Signature**

```python
def clean_postcode(raw: str, valid: set[str]) -> str | None
```

**Toy example**

```python
valid = load_valid_postcodes("data/valid_postcodes.txt")    # {'28001', ..., '28055'}
clean_postcode(" 28.013", valid)        # '28013'  (spaces and dots removed)
clean_postcode("Madrid 28013", valid)   # '28013'  (an E- or Madrid prefix removed, in any case)
clean_postcode("28099", valid)          # None     (28 and three digits, but not in valid)
clean_postcode("2813", valid)           # None
```

Return the postcode as **text**: as a number, `08013` would lose its leading zero.

**Why no regular expression in the cleaner?** After the clean-up, `text in valid` already checks the whole text,
digit by digit, against the 55 postcodes Garabato delivers to: nothing is left for a pattern to check. A regular
expression earns its place when you must FIND a postcode inside longer text (`Retiro (28009)`): that's what the
regex steps after the CORE do, with the gauntlet.

**Stuck for 7 minutes after Hint 2?** Run `clean_postcode = rescue("ENG1")` in a new cell, carry on, and come back to ENG1 later.

In [ ]:
# ===== YOUR CODE HERE =====
# ✏️ WRITE CODE: replace `pass` with your code, then run this cell (Shift+Enter).
def clean_postcode(raw: str, valid: set[str]) -> str | None:
    """Return a clean Madrid postcode such as "28013" that is in `valid`, or None."""
    # TODO 1: strip, then remove every space and every dot
    # TODO 2: drop an "E-" or "Madrid" prefix (compare in upper case)
    # TODO 3: return the rest if it is in valid (a set of postcodes); otherwise None
    pass

In [ ]:
# ▶️ JUST RUN · check your work
check("ENG1", clean_postcode);

<details>
<summary><strong>Hint 1 · Nudge</strong></summary>

Normalise first (`strip`, `replace`), then remove a prefix, then ask one question: is the rest one of the postcodes in `valid`? (A set answers `in` instantly, Theory S3.)

</details>

<details>
<summary><strong>Hint 2 · Plan</strong></summary>

1. `text = raw.strip().replace(" ", "").replace(".", "")`.
2. If `text.upper().startswith("E-")`: `text = text[2:]`; elif it starts with `"MADRID"`: `text = text[6:]`.
3. If `text in valid`: return `text`. Otherwise return `None`.

</details>

<details>
<summary><strong>Hint 3 · Ask an AI reviewer (keep only what you can explain)</strong></summary>

Copy this into your AI assistant: *"I'm learning Python. Act as a code reviewer and do NOT rewrite my code. Here is my function and what the check or my test printed: [paste]. Point out one problem at a time and ask me how I would fix it."*

</details>

In [ ]:
# ▶️ JUST RUN · YOUR clean_postcode on the postcode column of the export
valid_postcodes = load_valid_postcodes("data/valid_postcodes.txt")
if ready(clean_postcode):
    pairs = [(row["postcode"], clean_postcode(row["postcode"], valid_postcodes)) for row in rows]
    rejected = sorted({raw for raw, clean in pairs if clean is None})
    repaired = sorted({(raw, clean) for raw, clean in pairs if clean is not None and raw != clean})
    print(f"{sum(clean is not None for _, clean in pairs):,} of {len(pairs):,} postcodes are clean or repaired.")
    print("Some your cleaner repaired:", repaired[:6])
    print("Some that give None:       ", rejected[:6])

**Think about it.** A clean postcode also makes a second check possible: `postcode[int(box) - 1]` should be
the digit the labelling team typed. Why does that check need `clean_postcode` first? (Try it on a row whose
postcode is `28.033`.)

### OPTIONAL · SCI1 · Rescue scanner C · Scientist track · ✏️ WRITE CODE

Scanner C writes each pixel as a fraction from 0.0 to 1.0, so the CORE rejected every one of its scans. Rescue them: 0.4627 is round(0.4627 * 255) = 118.

**What you'll learn:** turn decimals into pixels, and measure which model suffers from which defect (Theory S2, Theory S6) · **Difficulty:** moderate · **Typical time:** 12–15 min

**Signature**

```python
def rescale_floats(raw: str) -> list[int] | None
```

**Toy example**

```python
rescale_floats("0.0 0.4627 1.0 ... 0.0")    # 784 fractions -> [0, 118, 255, ..., 0]
rescale_floats("0.5 ... 1.2")               # 1.2 is not a fraction from 0 to 1 -> None
rescale_floats("0 0 200 ... 0")             # 200 is above 1 (that's a scanner A row) -> None
```

**Stuck for 7 minutes after Hint 2?** Run `rescale_floats = rescue("SCI1")` in a new cell, carry on, and come back to SCI1 later.

In [ ]:
# ===== YOUR CODE HERE =====
# ✏️ WRITE CODE: replace `pass` with your code, then run this cell (Shift+Enter).
def rescale_floats(raw: str) -> list[int] | None:
    """Return 784 fractions from 0.0 to 1.0 as ints from 0 to 255 (round(v * 255)), or None."""
    # TODO: C2's plan with float() instead of int(): exactly 784 pieces, each one a float
    #       (ValueError -> None) from 0.0 to 1.0 (otherwise None); keep round(v * 255)
    pass

In [ ]:
# ▶️ JUST RUN · check your work
check("SCI1", rescale_floats);

<details>
<summary><strong>Hint 1 · Nudge</strong></summary>

It's C2 with `float()` instead of `int()`, a range of 0.0 to 1.0 instead of 0 to 255, and `round(v * 255)` for each value you keep. One trap: `float("NaN")` does NOT raise.

</details>

<details>
<summary><strong>Hint 2 · Plan</strong></summary>

1. `pieces = raw.split()`; not 784 of them: return `None`.
2. For each piece: `value = float(piece)` inside `try/except ValueError` (return `None` in the except block).
3. `if not 0.0 <= value <= 1.0: return None` (written this way, NaN fails the test too); otherwise append `round(value * 255)`.

</details>

<details>
<summary><strong>Hint 3 · Ask an AI reviewer (keep only what you can explain)</strong></summary>

Copy this into your AI assistant: *"I'm learning Python. Act as a code reviewer and do NOT rewrite my code. Here is my function and what the check or my test printed: [paste]. Point out one problem at a time and ask me how I would fix it."*

</details>

In [ ]:
# ▶️ JUST RUN · rescue scanner C: how many more rows does the pipeline keep? (about 1 s)
if ready(rescale_floats, clean_label, parse_pixels, fix_inversion):
    def parse_any(raw: str) -> list[int] | None:
        """C2's parse_pixels first; when it says None, try the scanner C fractions."""
        pixels = parse_pixels(raw)
        return pixels if pixels is not None else rescale_floats(raw)

    rescued_rows, rescued_report = run_pipeline(rows, clean_label, parse_any, fix_inversion)
    print(f"The CORE cleaning kept {len(clean_rows):,} rows; with scanner C rescued, {len(rescued_rows):,}.")

**Which model suffers from which defect?** The next cell measures two models on three training sets: the
average-digit model (28 × 28, the model of the incident) and nearest neighbour (Lab 1's reader, at 14 × 14
to keep it quick). It takes about 10 seconds; the first line says how long.

In [ ]:
# ▶️ JUST RUN · two models, three training sets (about 10 s)
if not ready(rescale_floats):
    pass                                  # ready() has said what is missing
elif not globals().get("rescued_rows"):
    print("Run the cell above first (it needs C1, C2, C3 and rescale_floats).")
else:
    if "naive_images" not in globals():   # the incident cell was skipped (or the kernel restarted)
        naive_images, naive_labels = naive_load("data/parcel_scans_raw.csv")
    table = model_table(naive=(naive_images, naive_labels), cleaned=clean_rows,
                        cleaned_and_rescued=rescued_rows)

### Explain your result (SCI1)

Copy your numbers into the table, then explain it in three sentences.

| Training set | Average digit | Nearest neighbour |
|---|---|---|
| naive | | |
| cleaned (CORE) | | |
| cleaned + scanner C rescued | | |

- Which model was hurt most by the naive data? (An average uses **every** image of a digit; nearest
  neighbour copies the label of **one** image: the closest.)
- Did rescuing scanner C help one model more than the other?
- The export also has a few well-formed but wrong labels (`"9"` typed under a 4). Which of the two models
  do you think copies that kind of mistake more readily, and why?

### OPTIONAL · BLD1 · Depot names · Builder track · ✏️ WRITE CODE

The depot column writes Chamartín in nine different ways. Map every spelling to the official name, or None for a depot that doesn't exist.

**What you'll learn:** Unicode: two ways to write í, mojibake, cp1252 and casefold; then a table per depot (Theory S6) · **Difficulty:** moderate · **Typical time:** 12–15 min

**Signature**

```python
def clean_depot(raw: str, aliases: dict[str, str]) -> str | None
```

**Toy example**

```python
clean_depot(" CHAMARTÍN ", aliases)      # 'Chamartín'  (strip, casefold, look up)
clean_depot("Chamarti\u0301n", aliases)  # 'Chamartín'  (i + a combining accent: normalise to NFC)
clean_depot("ChamartÃ\xadn", aliases)    # 'Chamartín'  (mojibake: UTF-8 read as Latin-1, repaired)
clean_depot("VLC", aliases)              # 'Vallecas'   (an alias)
clean_depot("Getafe", aliases)           # None
```

**Stuck for 7 minutes after Hint 2?** Run `clean_depot = rescue("BLD1")` in a new cell, carry on, and come back to BLD1 later.

> **New tool: three Unicode repairs**
>
> ```python
> import unicodedata
> unicodedata.normalize("NFC", text)        # i + combining accent (2 code points) -> í (1 code point)
> text.encode("latin-1").decode("utf-8")    # undo mojibake: 'ChamartÃ­n' -> 'Chamartín' (raises an error
>                                           # on text that isn't mojibake: catch UnicodeError, keep the text)
> text.casefold()                           # lower() for every alphabet: 'CHAMARTÍN' -> 'chamartín'
> ```

In [ ]:
# ▶️ JUST RUN · data/depots_reference.csv comes from an old Windows program: try UTF-8 first, then cp1252
try:
    with open("data/depots_reference.csv", encoding="utf-8", newline="") as f:
        f.read()
    print("As UTF-8: read without an error.")
except UnicodeDecodeError as error:
    print("As UTF-8: UnicodeDecodeError:", error)
aliases = {}
with open("data/depots_reference.csv", encoding="cp1252", newline="") as f:
    for row in csv.DictReader(f):
        aliases[row["alias"].strip().casefold()] = row["depot"].strip()
print("As cp1252:", aliases)

In [ ]:
# ===== YOUR CODE HERE =====
# ✏️ WRITE CODE: replace `pass` with your code, then run this cell (Shift+Enter).
import unicodedata


def clean_depot(raw: str, aliases: dict[str, str]) -> str | None:
    """Return the official depot name for any spelling in `aliases`, or None."""
    # TODO 1: strip; then try to repair mojibake (encode latin-1, decode utf-8) inside try/except UnicodeError
    # TODO 2: normalise to NFC, casefold, and look the result up in aliases (None if it isn't there)
    pass

In [ ]:
# ▶️ JUST RUN · check your work
check("BLD1", clean_depot);

<details>
<summary><strong>Hint 1 · Nudge</strong></summary>

Three repairs, then one lookup: strip, undo mojibake (only when it works: `try`), normalise to NFC, casefold, `aliases.get(...)`. The New tool box above has each line.

</details>

<details>
<summary><strong>Hint 2 · Plan</strong></summary>

1. `text = raw.strip()`.
2. `try: text = text.encode("latin-1").decode("utf-8")` and `except UnicodeError: pass` (not mojibake: keep it).
3. `text = unicodedata.normalize("NFC", text)`, then return `aliases.get(text.casefold())`: the official name, or None.

</details>

<details>
<summary><strong>Hint 3 · Ask an AI reviewer (keep only what you can explain)</strong></summary>

Copy this into your AI assistant: *"I'm learning Python. Act as a code reviewer and do NOT rewrite my code. Here is my function and what the check or my test printed: [paste]. Point out one problem at a time and ask me how I would fix it."*

</details>

In [ ]:
# ▶️ JUST RUN · rows per depot, as YOUR clean_depot reads them, and how many survived the CORE cleaning
depot_table(rows, clean_depot, aliases, clean_rows)

**Think about it.** Which depot lost the most rows in the CORE cleaning? Look at the scanner column: why?
What would you tell the scanner vendor?

## ADVANCED: for students who move faster

For students who move faster. These tasks are more open: a sanity check plus a cell where you explain your result. Nothing here is needed for the next lab.

| Task | What you'll learn | Difficulty | Typical time |
|---|---|---|---|
| **ENG2** Module and properties | your cleaners as a module of your own, and property tests on 200 tricky strings | demanding | 20–25 min |
| **SCI2** Find the liars | find well-formed but wrong labels with averages and nearest neighbours; precision and recall | demanding | 20–25 min |
| **BLD2** A report for humans | dates in four formats, a Markdown report, and redacting personal data | demanding | 20–25 min |
| **SCI3** Does a real neural network care? | train a scikit-learn neural network on the dirty and the clean data (needs scikit-learn) | moderate | 15–20 min |
| **Challenge** Batch B | run your pipeline unchanged on next week's export, then make it generalise; or compare your pipeline with pandas and DuckDB (Theory S14, S17) | open-ended | open-ended |

In [ ]:
# ▶️ JUST RUN · tools for the ADVANCED tasks (the same as at the start of OPTIONAL: run this once)
from labkit import load_valid_postcodes
from labkit.gauntlet import gauntlet
from labkit.lab3_tools import depot_table, find_suspects, show_suspects
from labkit.scans import model_table

### ADVANCED · ENG2 · Module and properties · Engineer track · ✏️ WRITE CODE

Your cleaners live in notebook cells. Move them into a module other programs can import, then test properties that must hold for ANY input, not just the cases you thought of.

**What you'll learn:** your cleaners as a module, and property tests on 200 tricky strings (Theory S13) · **Difficulty:** demanding · **Typical time:** 20–25 min

**Why a module?** A function that lives in a notebook cell can only be used by that notebook. Saved in a `.py`
file, it can be imported by any program: a second notebook, a test, or a command-line tool (Theory
S13: a module is a `.py` file you `import`).

**Step 1 · the module.** The kit has a file `cleaners.py` next to this notebook. Open it in VS Code, paste
your functions below their docstrings (`clean_label`, `parse_pixels`, and any OPTIONAL ones you wrote), follow
its three module rules (no `print()`, `None` and only `None` for "can't trust", a docstring in every function),
and save it. Then run the next two cells, and run the import cell again after every save: it reloads your
module, so the check sees your latest version and names the right line. The check also runs the C1 and C2
checks on your module, and calls
every cleaner with 200 tricky strings (`fuzz_strings(200)`: empty text, `NaN`, `²`, accents written two
ways, very long text, ...). A cleaner may return a value or `None`, but it must never crash.

**Step 2 · two property tests** in the notebook. A property is a rule that must hold for EVERY input, so you
don't need to know each right answer: `never_raises(fn, inputs)` returns the inputs that make `fn` raise, and
`not_idempotent(fn, inputs)` the inputs where cleaning twice differs from cleaning once (`fn(fn(x)) != fn(x)`).
The second one is for cleaners that return text they can read again: postcodes and depots.

In [ ]:
# ▶️ JUST RUN · import YOUR module, and reload it: run this cell again after every save of cleaners.py
import importlib
import cleaners
importlib.reload(cleaners);

In [ ]:
# ▶️ JUST RUN · check your work
check("ENG2", cleaners);

In [ ]:
# ===== YOUR CODE HERE =====
# ✏️ WRITE CODE: replace `pass` with your code, then run this cell (Shift+Enter).
def never_raises(fn, inputs: list[str]) -> list[str]:
    """Return the inputs for which fn(text) raised an exception (an empty list: the property holds)."""
    # TODO: call fn(text) for every text inside try/except Exception; collect the texts that raised
    pass


def not_idempotent(fn, inputs: list[str]) -> list[str]:
    """Return the inputs x for which fn(x) is not None and fn(fn(x)) != fn(x)."""
    # TODO: once = fn(text); keep the text when once is not None and fn(once) != once
    pass

<details>
<summary><strong>Hint 1 · Nudge</strong></summary>

A property test doesn't know the right answer for each input; it checks something that must be true for EVERY input. Loop over the inputs, call the cleaner, and collect the inputs that break the property.

</details>

<details>
<summary><strong>Hint 2 · Plan</strong></summary>

1. `never_raises`: `failures = []`; for each text: `try: fn(text)`, `except Exception: failures.append(text)`; return `failures`.
2. `not_idempotent`: for each text, `once = fn(text)`; if `once is not None and fn(once) != once`, append the text.
3. To test a cleaner with more parameters, wrap it: `lambda text: cleaners.clean_postcode(text, valid_postcodes)`.

</details>

<details>
<summary><strong>Hint 3 · Ask an AI reviewer (keep only what you can explain)</strong></summary>

Copy this into your AI assistant: *"I'm learning Python. Act as a code reviewer and do NOT rewrite my code. Here is my function and what the check or my test printed: [paste]. Point out one problem at a time and ask me how I would fix it."*

</details>

In [ ]:
# ▶️ JUST RUN · the two properties, for every cleaner of YOUR module, on 200 tricky strings
from labkit.check import is_stub
from labkit.fuzz import fuzz_strings

texts = fuzz_strings(200)
valid_postcodes = load_valid_postcodes("data/valid_postcodes.txt")
extra = {"clean_postcode": (valid_postcodes,), "clean_depot": (globals().get("aliases") or {},)}
if ready(never_raises, not_idempotent):
    for name in ("clean_label", "parse_pixels", "rescale_floats", "clean_postcode", "clean_depot"):
        fn = getattr(cleaners, name, None)
        if fn is None or is_stub(fn):
            print(f"{name:15} not in cleaners.py yet")
            continue
        args = extra.get(name, ())
        one = lambda text, fn=fn, args=args: fn(text, *args)
        broken = never_raises(one, texts)
        line = f"{name:15} raises on {len(broken)} of {len(texts)} inputs" + (f", e.g. {broken[0]!r}" if broken else "")
        if name in ("clean_postcode", "clean_depot") and not broken:
            unstable = not_idempotent(one, texts)
            line += f"; cleaning twice differs on {len(unstable)}"
        print(line)

### Explain your result (ENG2)

- Did any of your cleaners crash on a fuzz string? Which string, and why? (A classic: `"²".isdigit()` is
  True, but `int("²")` raises `ValueError`.)
- Why is "never crashes" a better property for a cleaner than "always returns a value"?
- Name one more property your cleaners should have, and how you would test it.

### ADVANCED · SCI2 · Find the liars · Scientist track · ✏️ WRITE CODE

A few rows have a well-formed label that is simply wrong: a 4 typed as '9'. No cleaner can see that from the text. Can the images give them away?

**What you'll learn:** find well-formed but wrong labels with averages and nearest neighbours; precision and recall · **Difficulty:** demanding · **Typical time:** 20–25 min

#### The idea, without code

**Stage 1 (fast, rough):** compute the average image of every label (as in Lab 1's SCI1), using the clean
rows. A row is *suspect* when the average of ANOTHER label is closer to its image than the average of its
own label.

**Stage 2 (slow, precise, only for the suspects):** look at the suspect's 5 nearest neighbours among the
other clean rows (at 14 × 14, with `downsample()`, so it takes seconds, not minutes). If none of them carries
the suspect's label, keep it on the list.

Many good rows are just unusual handwriting, so expect false alarms: the list is for a **person** to review,
who decides whether to relabel, drop or keep each row. The check scores your list against the rows whose
label really is wrong: **precision** (of your suspects, the share that really are wrong) and **recall** (of
the wrong rows, the share you found).

```python
def suspects(clean_rows: list[dict]) -> list[str]     # the scan_ids of the suspects
```

In [ ]:
# ===== YOUR CODE HERE =====
# ✏️ WRITE CODE: replace `pass` with your code, then run this cell (Shift+Enter).
from labkit import downsample


def suspects(clean_rows: list[dict]) -> list[str]:
    """Return the scan_ids whose typed label is probably wrong, for a person to review."""
    # TODO 1: the average image of each label (a dict of running totals and counts, then divide)
    # TODO 2: stage 1: the rows whose closest average belongs to another label
    # TODO 3: stage 2: for those rows only, the 5 nearest other rows at 14 x 14 (downsample);
    #         keep the row when none of them has its label; return the scan_ids
    pass

In [ ]:
# ▶️ JUST RUN · check your work
check("SCI2", suspects);   # runs it on the clean rows of Putting It All Together (5–15 s, depending on your Python)

<details>
<summary><strong>Hint 1 · Nudge</strong></summary>

Two loops you already know: Lab 1's averages (a dict of per-pixel running totals) and Lab 1's nearest neighbour, here with a sort over (distance, position) pairs to get the 5 closest.

</details>

<details>
<summary><strong>Hint 2 · Plan</strong></summary>

1. Averages: for each row, add its pixels into `totals[label]` (a list of 784 numbers) and count it; then divide.
2. Stage 1: for each row, the squared distance to every average; if the label of the smallest is not the row's label, remember its position.
3. Stage 2: `small = [downsample(row["pixels"]) for row in clean_rows]`; for each suspect `i`, build `(distance, j)` for every other `j`, sort, take the first 5; if none of their labels equals the suspect's label, keep `clean_rows[i]["scan_id"]`.

</details>

<details>
<summary><strong>Hint 3 · Ask an AI reviewer (keep only what you can explain)</strong></summary>

Copy this into your AI assistant: *"I'm learning Python. Act as a code reviewer and do NOT rewrite my code. Here is my function and what the check or my test printed: [paste]. Point out one problem at a time and ask me how I would fix it."*

</details>

In [ ]:
# ▶️ JUST RUN · your suspects among the clean rows of Putting It All Together, and four of them drawn (5–15 s; instant right after the check)
if ready(suspects) and clean_rows:
    my_suspects = find_suspects(suspects, clean_rows)    # reuses the check's answer if nothing changed
    if my_suspects is not None:
        print(f"{len(my_suspects)} suspects among {len(clean_rows):,} clean rows.")
        show_suspects(clean_rows, my_suspects, n=4)

### Explain your result (SCI2)

- How many suspects did you get, and what were the precision and recall in the check?
- Look at the drawn suspects: which would you relabel, which would you drop, which would you keep?
- Which costs Garabato more: a missed liar (low recall) or a person reviewing a false alarm (low
  precision)? How would you change stage 2 to move the balance?
- The images are one source of evidence; the `postcode` and `box` columns are another: the counter clerk
  typed the same digit, separately. How could `postcode[int(box) - 1]` confirm or clear a suspect, and what
  could go wrong with it?

### ADVANCED · BLD2 · A report for humans · Builder track · ✏️ WRITE CODE

The JSON report is for programs. The scanner vendor wants something a person can read, with dates in one format and no personal data.

**What you'll learn:** dates in four formats, a Markdown report, and redacting personal data · **Difficulty:** demanding · **Typical time:** 20–25 min

**Signature**

```python
def parse_timestamp(raw: str) -> str | None
def quality_report_md(report: dict, clean_rows: list[dict]) -> str
def redact(text: str) -> str            # optional
```

**Toy example**

```python
parse_timestamp("02/11/2026 14:05")    # '2026-11-02T14:05'  (day first: 2 November)
parse_timestamp("2026/11/02")          # '2026-11-02'        (a date only)
parse_timestamp("N/A")                 # None
quality_report_md(report, clean_rows)  # '# Scanner export: quality report\n\n...| Reason | Rows |...'
redact("call 07700 900461")            # 'call [PHONE]'
```

Two functions, one check: `check("BLD2", parse_timestamp)` checks `parse_timestamp` first, then `quality_report_md`. `redact` is optional: the cell after the check tries it on real notes.

In [ ]:
# ===== YOUR CODE HERE =====
# ✏️ WRITE CODE: replace `pass` with your code, then run this cell (Shift+Enter).
import re
from datetime import datetime

FORMATS = [("%Y-%m-%dT%H:%M", "%Y-%m-%dT%H:%M"),       # 2026-11-02T14:05 (already ISO 8601)
           ("%d/%m/%Y %H:%M", "%Y-%m-%dT%H:%M"),       # 02/11/2026 14:05 (day first)
           ("%Y/%m/%d", "%Y-%m-%d"),                    # 2026/11/02 (a date only)
           ("%d/%m/%Y", "%Y-%m-%d")]                    # 02/11/2026 (a date only, day first)
EMAIL = None        # optional, for redact: re.compile(r"...") (test it with gauntlet("email", pattern))
PHONE = None        # optional, for redact (test it with gauntlet("phone", pattern))


def parse_timestamp(raw: str) -> str | None:
    """Return the timestamp in ISO 8601 ("2026-11-02T14:05", or "2026-11-02" for a date only), or None."""
    # TODO: try each format with datetime.strptime (ValueError: try the next); none fits: None
    pass


def quality_report_md(report: dict, clean_rows: list[dict]) -> str:
    """Return the quality report as Markdown: a title, the totals, a table of reasons, rows kept per depot."""
    # TODO: build a list of lines ("# ...", "| Reason | Rows |", "|---|---:|", ...) and join them with "\n"
    pass


def redact(text: str) -> str:
    """Replace e-mail addresses with [EMAIL] and phone numbers with [PHONE]; scan ids and postcodes stay."""
    # TODO (optional): write EMAIL and PHONE above; then EMAIL.sub("[EMAIL]", text), and the same for PHONE
    pass

In [ ]:
# ▶️ JUST RUN · check your work
check("BLD2", parse_timestamp);   # checks parse_timestamp, then quality_report_md

<details>
<summary><strong>Hint 1 · Nudge</strong></summary>

`datetime.strptime(text, fmt)` either returns a datetime or raises ValueError: try the formats one by one in a loop. For the Markdown, build a list of lines and join them at the end.

</details>

<details>
<summary><strong>Hint 2 · Plan</strong></summary>

1. `parse_timestamp`: for each `(given, wanted)` in `FORMATS`: `try: return datetime.strptime(raw.strip(), given).strftime(wanted)`, `except ValueError: continue`; after the loop return `None`.
2. `quality_report_md`: `lines = ["# Scanner export: quality report", "", f"{report['valid']} of {report['total']} rows kept ..."]`, then a table header and one `| reason | count |` line per reason, then a table of rows per depot (count with a dict); `return "\n".join(lines)`.
3. `redact` (optional): write the two patterns first and test them with `gauntlet("email", pattern)` and `gauntlet("phone", pattern)`; then `EMAIL.sub("[EMAIL]", text)` replaces every match, and the same for `PHONE`.

</details>

<details>
<summary><strong>Hint 3 · Ask an AI reviewer (keep only what you can explain)</strong></summary>

Copy this into your AI assistant: *"I'm learning Python. Act as a code reviewer and do NOT rewrite my code. Here is my function and what the check or my test printed: [paste]. Point out one problem at a time and ask me how I would fix it."*

</details>

In [ ]:
# ▶️ JUST RUN · dates of the export in ISO 8601, YOUR report as output/quality_report.md, and redacted notes
from pathlib import Path
from labkit.check import is_stub
if ready(parse_timestamp):
    parsed = [parse_timestamp(row["scanned_at"]) for row in rows]
    print(f"{sum(p is not None for p in parsed):,} of {len(parsed):,} timestamps parsed.",
          "Not parsed:", sorted({row["scanned_at"] for row, p in zip(rows, parsed) if p is None})[:5])
if ready(quality_report_md) and clean_rows:
    named = clean_rows                     # with BLD1 done, the depots get their official names first
    if "clean_depot" in globals() and not is_stub(clean_depot) and globals().get("aliases"):
        named = [dict(row, depot=clean_depot(row["depot"], aliases) or row["depot"]) for row in clean_rows]
    text = quality_report_md(report, named)
    Path("output").mkdir(exist_ok=True)
    Path("output/quality_report.md").write_text(text, encoding="utf-8")
    print("Saved output/quality_report.md (open it in VS Code: right-click > Open Preview):\n")
    print(text)
if not is_stub(redact) and EMAIL is not None and PHONE is not None:
    notes = [row["notes"] for row in rows if "@" in row["notes"] or "555-" in row["notes"]
             or "07700" in row["notes"]][:5]
    for note in notes:
        print(f"{note!r}\n   -> {redact(note)!r}")

### Explain your result (BLD2)

- Which timestamps did not parse, and should they? What would you report to the vendor about dates?
- Is "02/11/2026" the 2nd of November or the 11th of February? How did you decide, and how would you make
  sure next time?
- Did `redact` leave anything personal behind, or remove something it shouldn't (a scan id, a postcode)?

### ADVANCED · SCI3 · Does a real neural network care? · Scientist track · ✏️ WRITE CODE

The average-digit model is simple. Would a real neural network, which learns its own features from the pixels, shrug off the dirty data?

**What you'll learn:** train a scikit-learn neural network on the dirty and the clean data (needs scikit-learn) · **Difficulty:** moderate · **Typical time:** 15–20 min

**Signature**

```python
def compare_training_sets(make_model, training_sets: dict, test_images: list[list[int]], test_labels: list[int]) -> dict[str, float]
```

**Toy example**

```python
compare_training_sets(make_model, {"naive": (naive_images, naive_labels), "cleaned": (images, labels)},
                      test_images, test_labels)
# {'naive': ..., 'cleaned': ...}: the test accuracy of a NEW model trained on each set
```

**What you'll write** works with any model that has `fit(X, y)` and `score(X, y)`, like every scikit-learn
model: make a NEW model for each training set (`make_model()`), scale the pixels to 0.0-1.0 (`p / 255`, for the
training AND the test images), train it, score it on the test images. The check uses a stand-in model, so it
works without scikit-learn.

**The experiment** (the cell after the check) needs scikit-learn, an optional library: `uv sync --group nn`
in the kit folder (or `pip install scikit-learn`), or Google Colab. Without it, the cell says so and skips.

In [ ]:
# ===== YOUR CODE HERE =====
# ✏️ WRITE CODE: replace `pass` with your code, then run this cell (Shift+Enter).
def compare_training_sets(make_model, training_sets: dict, test_images: list[list[int]],
                          test_labels: list[int]) -> dict[str, float]:
    """Train a NEW model on each training set; return {name: accuracy on the test images}."""
    # TODO 1: scale the test images: every pixel / 255 (new lists)
    # TODO 2: for each name, (images, labels) in training_sets.items(): model = make_model(),
    #         scale the images, model.fit(X, labels), then model.score(X_test, test_labels)
    # TODO 3: return {name: score}
    pass

In [ ]:
# ▶️ JUST RUN · check your work
check("SCI3", compare_training_sets);   # with a stand-in model: no scikit-learn needed

<details>
<summary><strong>Hint 1 · Nudge</strong></summary>

Every scikit-learn model has the same two methods: `fit(X, y)` learns, `score(X, y)` returns the accuracy. Your function is a loop over the training sets that calls them, with a fresh model each time.

</details>

<details>
<summary><strong>Hint 2 · Plan</strong></summary>

1. `X_test = [[p / 255 for p in image] for image in test_images]`.
2. For each `name, (images, labels)` in `training_sets.items()`: `model = make_model()`, `X = [[p / 255 for p in image] for image in images]`, `model.fit(X, labels)`, `scores[name] = model.score(X_test, test_labels)`.
3. Return `scores`.

</details>

<details>
<summary><strong>Hint 3 · Ask an AI reviewer (keep only what you can explain)</strong></summary>

Copy this into your AI assistant: *"I'm learning Python. Act as a code reviewer and do NOT rewrite my code. Here is my function and what the check or my test printed: [paste]. Point out one problem at a time and ask me how I would fix it."*

</details>

In [ ]:
# ▶️ JUST RUN · a real neural network (scikit-learn's MLPClassifier) on the naive and the clean data (about 10 s; up to 30 s the first time, while scikit-learn loads)
import warnings
from labkit.optional import need

sklearn = need("sklearn", "the SCI3 experiment")
if sklearn is not None and ready(compare_training_sets) and clean_rows:
    from sklearn.exceptions import ConvergenceWarning
    from sklearn.neural_network import MLPClassifier

    def make_model():
        # 64 hidden units; 20 passes over the data (epochs), stopped early on purpose to stay quick
        return MLPClassifier(hidden_layer_sizes=(64,), max_iter=20, random_state=2026)

    test_images, test_labels = load_digits("data/mnist_test_2k.csv.gz")
    if "naive_images" not in globals():   # the incident cell was skipped (or the kernel restarted)
        naive_images, naive_labels = naive_load("data/parcel_scans_raw.csv")
    with warnings.catch_warnings():
        # scikit-learn warns that 20 passes are not enough to converge: we know, and we accept it
        warnings.simplefilter("ignore", ConvergenceWarning)
        network = compare_training_sets(make_model, {"naive": (naive_images, naive_labels),
                                                     "cleaned": as_training_set(clean_rows)},
                                        test_images, test_labels)
    averages = globals().get("results") or {}
    print(f"{'training set':<14}{'average digit':>15}{'neural network':>16}")
    for name, accuracy in network.items():
        average = f"{averages[name]:.1%}" if name in averages else "-"
        print(f"{name:<14}{average:>15}{accuracy:>16.1%}")

### Explain your result (SCI3)

- Did the neural network care about the dirty data? More or less than the average-digit model?
- Did it cope with the inverted and blank images better or worse than the average-digit model? What might
  explain that? (Think about what one average image per digit can represent, and what 64 hidden units
  can.)
- Is "a bigger model will fix bad data" a good plan for Garabato? What does your table say?

### ADVANCED · Challenge: Batch B: don't overfit your cleaners · ✏️ WRITE CODE

**What you'll learn:** making a pipeline generalise to data it has never seen, and measuring it honestly · **Difficulty:** open-ended · **Typical time:** open-ended (you can continue at home)

Next week's export has arrived: `data/holdout/parcel_scans_batch_b.csv`. First run your pipeline **unchanged** on it and score it with `score_batch_b(clean_rows, report)`: precision and recall of your rejections against the truth, and the accuracy of the average-digit model retrained on your clean rows. Then improve your pipeline so it **generalises**, and score it again. AI is your pair programmer here: verify what it suggests, and say how you used it in your Engineering log.

**Ideas to try**

- Read before you clean: look at the first lines of batch B (`show_file`), then at the rows your pipeline rejects, reason by reason (`show_rows`). Which rejections deserve a repair instead?
- Keep the CORE order of the rules; change how each rule reads its value.
- Stuck? The pointers below the step 1 cell open one at a time.

**Rules**

- No special cases for scan ids or line numbers: every rule must work on next week's export too.
- Standard library only (the pandas and DuckDB alternative below is the exception).
- Score each idea once with `score_batch_b`, and write down what generalised and what didn't.
- **Batch C plays the role of the test images:** the export of the week after, which your rules were never tuned on. Don't open `data/holdout/parcel_scans_batch_c.csv`; run step 3 once, at the end.

**Or choose an alternative**

- **Compare with the tools of Theory S14 and S17** (pandas and DuckDB): what do they decide for you when they read the export, can your own rules run inside pandas, and which tool would you use for which job? Optional libraries; cells below.
- **A streaming pipeline:** a generator that reads and cleans one row at a time, compared with the list version on a big export made by `make_big_scans()`, with `tracemalloc` measuring the memory. Cells below.

In [ ]:
# ▶️ JUST RUN · step 1: YOUR pipeline, unchanged, on batch B, then the score
from labkit.scans import score_batch_b
rows_b, bad_b = read_scans("data/holdout/parcel_scans_batch_b.csv")
print(f"{len(rows_b)} rows read; the keys of the first row: {list(rows_b[0])[:3]} ...")
if ready(clean_scans):
    try:
        clean_b, report_b = clean_scans(rows_b)
        score_b = score_batch_b(clean_b, report_b)
    except Exception as error:
        print(f"Your pipeline stopped with {type(error).__name__}: {error}")
        print("Look closely at the keys printed above.")

<details>
<summary><strong>Pointer 1 · My pipeline stops before it cleans a single row</strong></summary>

Look at the keys of the first row printed above: what is in front of `scan_id`? Search for "byte-order
mark" and for the encoding `utf-8-sig`.

</details>

<details>
<summary><strong>Pointer 2 · It runs, but the precision is low</strong></summary>

Many of your rejections didn't deserve it. List the rows your pipeline rejects for `bad label`, then for
`bad pixels`, and read their values with `show_rows`: which could be repaired instead? Compare the
scanners too.

</details>

<details>
<summary><strong>Pointer 3 · Still stuck</strong></summary>

Labels in another language, scanner C's fractions (SCI1), a new scanner D: which of them is a reason to
reject, and which a reason to repair? A repair must be checked too.

</details>

In [ ]:
# ===== YOUR CODE HERE =====
# ✏️ WRITE CODE: replace `pass` with your code, then run this cell (Shift+Enter).
# the imports and constants you need go here


def read_scans_b(path: str) -> tuple[list[dict[str, str]], list[int]]:
    """Like read_scans, for exports from any of Garabato's scanners."""
    # TODO: start from read_scans; what does batch B need?
    pass


def clean_scans_b(rows: list[dict[str, str]]) -> tuple[list[dict], dict]:
    """The CORE rules (same order, same report), reading every value more carefully."""
    # TODO: start from your clean_scans; change how each rule READS its value, not the rules
    pass

In [ ]:
# ▶️ JUST RUN · step 2: score YOUR improved pipeline on batch B
if ready(read_scans_b, clean_scans_b):
    rows_b2, bad_b2 = read_scans_b("data/holdout/parcel_scans_batch_b.csv")
    clean_b2, report_b2 = clean_scans_b(rows_b2)
    score_b2 = score_batch_b(clean_b2, report_b2)

**Step 3 · did it generalise? (once, at the end)** Batch C is the export of the week after: same scanners,
other scans. Your rules were never tuned on it (just as a model never trains on its test images), so its
score is an honest measure of your improvements. Compare it with step 2, and write the comparison in your Engineering log.

In [ ]:
# ▶️ JUST RUN · step 3: YOUR improved pipeline on batch C, scored once
from labkit.scans import score_batch_c
if ready(read_scans_b, clean_scans_b):
    rows_c, bad_c = read_scans_b("data/holdout/parcel_scans_batch_c.csv")
    clean_c, report_c = clean_scans_b(rows_c)
    score_c = score_batch_c(clean_c, report_c)

#### Alternative: compare your pipeline with pandas and DuckDB (Theory S14, S17)

Theory S14 introduces **pandas** (a table as a DataFrame) and **DuckDB** (SQL straight on a file), and S17
uses them for data analysis. If you haven't met them yet, keep this alternative for after S14. In a real
project you would reach for them first. So: do they make the scanner mess go away? Three short steps, then
you decide which tool you'd use for which job.

Both are optional libraries, never needed for the CORE: pandas with `uv sync --group dash` in the kit folder,
DuckDB with `%pip install duckdb` in a notebook cell (or `%pip install pandas duckdb` for both), then restart
the kernel. Google Colab already has pandas. Without them, each cell says so in one line and skips.

In [ ]:
# ▶️ JUST RUN · step A: pandas reads the export with its default settings
from labkit.optional import need
pd = need("pandas", "the pandas and DuckDB comparison")
if pd is not None:
    try:
        pd.read_csv("data/parcel_scans_raw.csv")
        print("pd.read_csv with its defaults: read without an error.")
    except Exception as error:
        print(f"pd.read_csv with its defaults: {type(error).__name__}: {str(error).strip()}")
    df_default = pd.read_csv("data/parcel_scans_raw.csv", on_bad_lines="skip")
    print(f"\nWith on_bad_lines=\"skip\": {len(df_default):,} rows. Values pandas read as missing (NaN):")
    print(df_default[["label_typed", "scanned_at"]].isna().sum().to_string())

**Look before you trust (1 minute).** How many rows did pandas keep, and can you tell which lines it dropped?
What did it do with the empty and the `n/a` labels, and the `N/A` dates? Is a `NaN` label a bad label, or a
label nobody typed?

<details>
<summary><strong>What pandas decided for you (open after you've run it)</strong></summary>

- With its defaults, `read_csv` stops at the first malformed line. With `on_bad_lines="skip"` it drops those
  lines **silently**; `read_scans` kept their line numbers, so you could report them to the vendor.
- An empty value and `N/A` (or `n/a`) both became `NaN` ("missing"): pandas guessed what they mean. Your
  `clean_label` had to decide, and the decision is written down in your code: a label you can't read is a
  bad label.
- `dtype=str, keep_default_na=False` tells pandas to keep every value as the text in the file, as
  `csv.DictReader` does. Step B starts from there.

</details>

> **New tool: the pandas pieces of step B (Theory S17)**
>
> | Piece | Means |
> |---|---|
> | `df["scan_id"].str.strip().str.upper()` | a string method on every value of a column |
> | `series.duplicated()` | `True` for the second and later copies of a value |
> | `series.map(clean_label)` | calls YOUR function on every value: your cleaners work unchanged |
> | `series.isna()` | `True` where the value is missing (`None` or `NaN`) |
> | `mask_a & mask_b`, `~mask` | and, not: on whole columns of `True` / `False` |
> | `df[mask]`, `df.loc[mask, ["scan_id", "depot"]]` | the rows where the mask is `True` (and some columns) |
> | `series.value_counts()` | how many times each value appears: a counting dict in one line |

In [ ]:
# ===== YOUR CODE HERE =====
# ✏️ WRITE CODE: replace `pass` with your code, then run this cell (Shift+Enter).
def clean_scans_pandas(df):
    """The three CORE rules on a DataFrame of text (one row per scan); return (clean DataFrame, report)."""
    # TODO 1: one True/False Series per rule: duplicate (.duplicated() on the normalised ids), and YOUR
    #         cleaners with .map(): labels = df["label_typed"].map(clean_label),
    #         pixels = df["pixels"].map(parse_pixels) (then .isna() says where they returned None)
    # TODO 2: the reason of each row: the FIRST rule it breaks ("" if it breaks none)
    # TODO 3: the clean DataFrame (scan_id, depot, label, pixels through fix_inversion) and the C4 report
    pass

In [ ]:
# ▶️ JUST RUN · step B: the pandas version next to YOUR clean_scans, on the same export
pd = need("pandas", "the pandas and DuckDB comparison")
if pd is not None and ready(clean_scans_pandas, clean_scans):
    df = pd.read_csv("data/parcel_scans_raw.csv", dtype=str, keep_default_na=False, on_bad_lines="skip",
                     encoding="utf-8")
    clean_pd, report_pd = clean_scans_pandas(df)
    clean_mine, report_mine = clean_scans(rows)
    print("pandas:", report_pd)
    print("yours: ", report_mine)
    same_rows = list(clean_pd["scan_id"]) == [row["scan_id"] for row in clean_mine]
    print(f"Same report: {report_pd == report_mine} · the same scans kept, in the same order: {same_rows}")

In [ ]:
# ▶️ JUST RUN · step C: DuckDB, one SQL query on the raw file: every way depot is written, per scanner
duckdb = need("duckdb", "the DuckDB queries")
if duckdb is not None:
    duckdb.sql("""
        SELECT scanner, depot, COUNT(*) AS n_rows
        FROM read_csv('data/parcel_scans_raw.csv', all_varchar = true, ignore_errors = true)
        GROUP BY scanner, depot
        ORDER BY scanner, n_rows DESC
    """).show(max_rows=40)

In [ ]:
# ===== YOUR CODE HERE =====
# ✏️ WRITE CODE · your query on output/scans_clean.csv (C5 wrote it): how many clean rows per label? Is it balanced?
from pathlib import Path
duckdb = need("duckdb", "the DuckDB queries")
if duckdb is not None and not Path("output/scans_clean.csv").exists():
    print("Run C5 first: it writes output/scans_clean.csv.")
elif duckdb is not None:
    query = "SELECT scan_id, depot, label FROM 'output/scans_clean.csv' LIMIT 5"     # a first query: replace it
    duckdb.sql(query).show()

**Which tool for which job?** (Write your answers in the Engineering log.)

- pandas called YOUR `clean_label` and `parse_pixels`. What did pandas do for you, and what did it still
  need from you?
- Which version reads more clearly to you: your loop, or the columns? With which one would you rather find
  the bug when one row goes wrong?
- When would you reach for DuckDB or pandas in this project, and when for plain Python functions? (Lab 4
  turns the digit reader into a command-line tool that Ops run on any laptop.)

#### Alternative: a streaming pipeline (ADVANCED, standard library)

`read_scans` keeps every row in a list before cleaning. A **generator** (a function with `yield`) can clean
one row at a time, so the memory it needs doesn't grow with the file. Write `stream_clean(path)`, then the
next cell builds a bigger export with `make_big_scans()` (3,000 rows here; try 100,000 at home) and
measures the peak memory of both ways with `tracemalloc`.

In [ ]:
# ===== YOUR CODE HERE =====
# ✏️ WRITE CODE: replace `pass` with your code, then run this cell (Shift+Enter).
import csv


def stream_clean(path: str):
    """Yield the clean rows of the file one at a time (the CORE rules), never holding the whole file."""
    # TODO: open the file, loop over csv.DictReader, apply the rules, and yield each clean row
    pass

In [ ]:
# ▶️ JUST RUN · all at once vs streaming, on a bigger export (about 10 s)
import tracemalloc
from labkit.scans import make_big_scans

if ready(stream_clean, clean_scans):
    big = make_big_scans("output/big_scans.csv", n=3_000)
    tracemalloc.start()
    all_rows, _ = read_scans(str(big))
    kept_list, _ = clean_scans(all_rows)
    list_peak = tracemalloc.get_traced_memory()[1]
    del all_rows, kept_list
    tracemalloc.reset_peak()
    kept_stream = sum(1 for _ in stream_clean(str(big)))
    stream_peak = tracemalloc.get_traced_memory()[1]
    tracemalloc.stop()
    print(f"all at once: peak {list_peak / 1e6:.1f} MB · streaming: peak {stream_peak / 1e6:.1f} MB "
          f"({kept_stream:,} rows kept)")

### Engineering log

Short notes are fine. Edit this cell as you go.

- **What I tried:**
- **What I measured:**
- **What I'd do next:**

## AI-use declaration (about 1 minute)

This course asks you to use AI critically and to say how you used it. This cell stays in your notebook:
nobody collects or grades it. Please don't write your name.

- **Tools I used** (or "none"):
- **For which tasks** (e.g. C2, ENG1):
- **One prompt I used** (copy it, or summarise it in one line):
- **How I checked the answer** (check(), my own test, reading the code, a classmate…):
- **Something the AI got wrong, or that I changed:**
- **One line of AI-written code I can now explain in my own words:**

## Before you go (private, 30 seconds)
- One thing I can do now that I couldn't at the start of this lab:
- One thing I'm still unsure about:

## What's next

- **Catch-up pack**, on IE Campus after class: `lab3_solution.ipynb` (CORE and OPTIONAL solutions), `lab3_express.ipynb` (the minimum path in about 25 minutes, with the same checks) and `lab3_what_you_missed` (a one-page summary).
- **ADVANCED solutions** (including the Challenge) come with the Lab 4 kit, in `solutions_previous/`, so the Challenge stays open until then.
- **Next:** Lab 4 · Ship It to the Depot, Wed 18 Nov, 15:00–16:20, MMB-303. The kit appears on IE Campus 48 hours before.
- **The League** is opt-in and uses pseudonyms: the rules are in `LEAGUE.md`.
- **Missed something?** `labkit/reference/lab3.py` has a clear, commented version of every CORE and OPTIONAL function of this lab (`lab1.py` and `lab2.py` next to it cover the earlier labs).
- **Lab 2's ADVANCED solutions**, the Challenge included, are in `solutions_previous/lab2_solution_full.ipynb`.
- **Your files:** if you did C5, `output/scans_clean.csv` and `output/quality_report.json` stay in your kit folder. Lab 4 turns the digit reader into a command Ops can run without Jupyter; today's habits (an encoding on every file, JSON, `None` for input you can't trust) come with you.